# Эксперимент: LLM-диагностика цифровой зрелости малого предприятия

**Цель.** Сравнить, насколько оценки цифровой зрелости малого предприятия, выполненные разными LLM по досье (только открытые источники).

- **Независимые переменные:** модель LLM и предоставленное ей досье.
- **Зависимая переменная:** уровень цифровой зрелости по пятиуровневой порядковой шкале.
- **Эталон:** независимая экспертная оценка по той же шкале.
- **Критерии приёмлемости:** κ ≥ 0,60; доля ошибок не более чем на один уровень ≥ 90 %; доля выдуманных фактов ≤ 5 %.

В ноутбуке зафиксированы: (1) единый промпт для всех моделей и протокол прогонов; (2) результаты эксперимента — три CSV-файла, встроенные в файл напрямую.


## 1. Промпт эксперимента (версия 1.0 — зафиксирован до прогонов)

Один и тот же текст для всех моделей. Заменитель: `{{ДОСЬЕ}}` — текст досье предприятия.


---

Ты — независимый эксперт по цифровой трансформации предприятий. Оцени цифровой уровень цифровизации малого предприятия по предоставленному досье, составленному ТОЛЬКО из открытых источников (сайт, карты/справочники, вакансии, открытые реестры, соцсети).

Шкала цифровой зрелости (пороговая: каждый уровень включает признаки предыдущего):
- 1 «Начальный»: единых цифровых процессов нет; эпизодическое использование почты/мессенджеров; сайта нет или он устарел; продажи и документооборот — офлайн.
- 2 «Стандартизация»: онлайн-присутствие (сайт, карточки в картах/справочниках, каталог); онлайн-оплата; частичная автоматизация (бухгалтерия, email-рассылки).
- 3 «Цифровизация»: онлайн-заказ/запись/электронная коммерция; CRM/учётная система; интеграция с доставкой, банком, маркетплейсами; сквозные цифровые процессы.
- 4 «Сетевой»: платформенность (маркетплейсы, экосистемы); решения на основе данных (аналитика, программы лояльности); интеграция клиентских и товарных данных.
- 5 «Интеллектуальный»: AI-инструменты в операционной деятельности (рекомендации, прогнозирование, интеллектуальные ассистенты); персонализация и адаптивные процессы.

Правила:
1. Опирайся ТОЛЬКО на факты, явно присутствующие в досье. Если признак не подтверждён досьем — не считай его имеющимся.
2. Не домысливай и не экстраполируй: «не упомянуто» = «не подтверждено».
3. Для каждого утверждения укажи ссылку на фрагмент досье (цитата ≤30 слов).
4. Если данных недостаточно для уверенной оценки уровня, так и укажи в confidence.

Досье:
"""
{{ДОСЬЕ}}
"""

Ответь СТРОГО в формате JSON без пояснений:
{
  "level": <целое 1–5, общий уровень>,
  "criteria": {
    "online_presence": <1–5>,
    "process_automation": <1–5>,
    "ecommerce_online_services": <1–5>,
    "data_driven": <1–5>,
    "ai_usage": <1–5>
  },
  "evidence": [
    {"claim": "<утверждение>", "quote": "<цитата из досье, ≤30 слов>"}
  ],
  "confidence": <число 0–1>,
  "level_rationale": "<обоснование уровня ≤50 слов>"
}

---

## 1.1. Замечания по протоколу

- temperature = 0 (где доступно), 3 независимых прогона на досье → для метрики стабильности (медиана + разброс).
- Поле `criteria` фиксировано: 5 критериев; для κ по критериям — среднее по 5.
- JSON-валидация: если ответ не парсится — прогон считается «сбой форматирования» (отдельная строка в табл. 3).
- Счётчик стоимости: токены (in/out) × тариф; для локальных моделей — время инференса на фиксированном железе.

## 2. Результаты эксперимента (CSV-файлы)

Данные встроены в ноутбук (ячейки ниже читают встроенные строки, файлы на диске не требуются).

| Файл | Переменная | Содержимое |
|---|---|---|
| `summary.csv` | `df_summary` | Все прогоны (включая сбои форматирования `format_fail` / `no_json`), оценки по критериям, токены, время |
| `summary_clean.csv` | `df_summary_clean` | Только успешные прогоны (`status = ok`), + колонка `level_rationale` |
| `summary_comprehensive.csv` | `df_summary_comprehensive` | Расширенная валидация: флаги (пустой evidence, нулевые критерии), счётчики ошибок и предупреждений |


In [1]:
import io

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
# summary.csv — Все прогоны (включая сбои форматирования)
# CSV встроено напрямую в файл ноутбука

_CSV_DF_SUMMARY_BODY = r"""model,dossier,run,status,parse_ok,ok,level,online_presence,process_automation,ecommerce_online_services,data_driven,ai_usage,confidence,n_evidence,quote_not_found,errors,attempts,total_s,prompt_tokens,completion_tokens,error
M10-GLM-4.7-Flash-Q6_K,01-ponchik,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,80.49,939,4096,
M10-GLM-4.7-Flash-Q6_K,01-ponchik,2,ok,True,True,3,3,3,5,3,1,0.9,4,1,,1,54.67,939,2873,
M10-GLM-4.7-Flash-Q6_K,01-ponchik,3,ok,True,True,3,3,3,5,3,1,0.9,4,1,,1,54.71,939,2873,
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,1,ok,True,True,2,2,2,2,1,1,0.9,3,1,,1,48.56,775,2555,
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,2,ok,True,True,2,4,1,3,1,1,0.4,3,1,,1,65.84,775,3450,
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,3,ok,True,True,2,4,1,3,1,1,0.4,3,1,,1,65.86,775,3450,
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,1,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,79.54,925,4096,
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,2,ok,True,True,3,3,3,4,3,1,0.9,4,1,,1,73.68,925,3827,
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,3,ok,True,True,3,3,3,4,3,1,0.9,4,1,,1,73.7,925,3827,
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,1,ok,True,True,2,4,1,2,1,1,0.8,4,0,,1,45.47,789,2398,
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,2,ok,True,True,2,3,2,2,2,1,0.9,4,1,,1,53.58,789,2827,
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,3,ok,True,True,2,3,2,2,2,1,0.9,4,1,,1,53.55,789,2827,
M10-GLM-4.7-Flash-Q6_K,05-moncafe,1,ok,True,True,2,2,2,2,1,1,0.9,4,1,,1,37.55,830,1984,
M10-GLM-4.7-Flash-Q6_K,05-moncafe,2,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,55.38,830,2916,
M10-GLM-4.7-Flash-Q6_K,05-moncafe,3,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,55.39,830,2916,
M10-GLM-4.7-Flash-Q6_K,06-breakfast-band,1,ok,True,True,2,2,2,2,2,1,0.8,4,0,,1,55.96,838,2926,
M10-GLM-4.7-Flash-Q6_K,06-breakfast-band,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.89,838,4096,
M10-GLM-4.7-Flash-Q6_K,06-breakfast-band,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.92,838,4096,
M10-GLM-4.7-Flash-Q6_K,07-ru-service,1,ok,True,True,2,2,2,1,1,1,0.85,4,1,,1,60.91,753,3182,
M10-GLM-4.7-Flash-Q6_K,07-ru-service,2,ok,True,True,2,4,2,2,2,1,0.8,3,0,,1,49.14,753,2602,
M10-GLM-4.7-Flash-Q6_K,07-ru-service,3,ok,True,True,2,4,2,2,2,1,0.8,3,0,,1,49.22,753,2602,
M10-GLM-4.7-Flash-Q6_K,08-avtobat,1,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,41.52,784,2190,
M10-GLM-4.7-Flash-Q6_K,08-avtobat,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.65,784,4096,
M10-GLM-4.7-Flash-Q6_K,08-avtobat,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.68,784,4096,
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.46,1023,4096,
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,2,ok,True,True,3,5,4,5,3,1,0.85,4,1,,1,68.22,1023,3628,
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,3,ok,True,True,3,5,4,5,3,1,0.85,4,1,,1,68.17,1023,3628,
M10-GLM-4.7-Flash-Q6_K,10-krizanta,1,ok,True,True,2,3,2,2,2,1,0.7,3,1,,1,46.25,775,2440,
M10-GLM-4.7-Flash-Q6_K,10-krizanta,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.62,775,4096,
M10-GLM-4.7-Flash-Q6_K,10-krizanta,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.64,775,4096,
M10-GLM-4.7-Flash-Q6_K,11-u-nails,1,ok,True,True,2,3,2,2,2,1,0.8,3,1,,1,65.69,795,3424,
M10-GLM-4.7-Flash-Q6_K,11-u-nails,2,ok,True,True,3,3,2,3,2,1,0.8,4,1,,1,68.3,795,3574,
M10-GLM-4.7-Flash-Q6_K,11-u-nails,3,ok,True,True,3,3,2,3,2,1,0.8,4,1,,1,68.31,795,3574,
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,1,ok,True,True,3,3,2,3,2,1,0.8,4,1,,1,67.65,757,3528,
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,2,ok,True,True,3,4,3,3,2,1,0.6,3,0,,1,51.87,757,2744,
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,3,ok,True,True,3,4,3,3,2,1,0.6,3,0,,1,51.83,757,2744,
M10-GLM-4.7-Flash-Q6_K,13-salon-radul,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,79.16,1050,4096,
M10-GLM-4.7-Flash-Q6_K,13-salon-radul,2,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,77.47,1050,4096,
M10-GLM-4.7-Flash-Q6_K,13-salon-radul,3,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,77.43,1050,4096,
M10-GLM-4.7-Flash-Q6_K,14-milliard-roz,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,77.84,984,4096,
M10-GLM-4.7-Flash-Q6_K,14-milliard-roz,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,77.32,984,4096,
M10-GLM-4.7-Flash-Q6_K,14-milliard-roz,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,77.29,984,4096,
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,1,ok,True,True,1,2,1,1,1,1,0.9,3,0,,1,57.86,952,3013,
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,2,ok,True,True,1,2,1,1,1,1,0.95,3,0,,1,62.52,952,3272,
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,3,ok,True,True,1,2,1,1,1,1,0.95,3,0,,1,62.58,952,3272,
M10-GLM-4.7-Flash-Q6_K,16-kameliya,1,ok,True,True,3,5,3,5,3,1,0.9,5,0,,1,74.98,1119,3850,
M10-GLM-4.7-Flash-Q6_K,16-kameliya,2,ok,True,True,3,5,4,5,3,1,0.95,4,0,,1,45.07,1119,2375,
M10-GLM-4.7-Flash-Q6_K,16-kameliya,3,ok,True,True,3,5,4,5,3,1,0.95,4,0,,1,45.01,1119,2375,
M10-GLM-4.7-Flash-Q6_K,17-ip-electronics,1,ok,True,True,2,3,2,2,2,1,0.9,4,1,,1,53.67,1114,2778,
M10-GLM-4.7-Flash-Q6_K,17-ip-electronics,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,79.26,1114,4096,
M10-GLM-4.7-Flash-Q6_K,17-ip-electronics,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,79.24,1114,4096,
M10-GLM-4.7-Flash-Q6_K,18-service69,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,79.43,958,4096,
M10-GLM-4.7-Flash-Q6_K,18-service69,2,ok,True,True,2,4,2,1,2,1,0.9,4,1,,1,47.11,958,2486,
M10-GLM-4.7-Flash-Q6_K,18-service69,3,ok,True,True,2,4,2,1,2,1,0.9,4,1,,1,47.05,958,2486,
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,1,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,62.06,1102,3201,
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,2,ok,True,True,2,2,2,2,1,1,0.8,4,1,,1,51.63,1102,2711,
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,3,ok,True,True,2,2,2,2,1,1,0.8,4,1,,1,51.6,1102,2711,
M10-GLM-4.7-Flash-Q6_K,20-agym,1,ok,True,True,3,5,4,4,2,1,0.9,4,0,,1,43.88,1039,2287,
M10-GLM-4.7-Flash-Q6_K,20-agym,2,ok,True,True,4,5,4,5,4,1,0.9,4,0,,1,60.32,1039,3178,
M10-GLM-4.7-Flash-Q6_K,20-agym,3,ok,True,True,3,5,4,5,2,1,0.9,5,1,,1,45.57,1039,2458,
M10-GLM-4.7-Flash-Q6_K,21-my-fit,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.36,1012,4096,
M10-GLM-4.7-Flash-Q6_K,21-my-fit,2,ok,True,True,3,4,4,3,3,1,0.85,5,1,,1,46.25,1012,2492,
M10-GLM-4.7-Flash-Q6_K,21-my-fit,3,ok,True,True,3,4,4,3,3,1,0.85,5,1,,1,46.17,1012,2492,
M10-GLM-4.7-Flash-Q6_K,22-lizavet,1,ok,True,True,2,2,2,2,2,1,0.8,4,0,,1,70.38,834,3656,
M10-GLM-4.7-Flash-Q6_K,22-lizavet,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.84,834,4096,
M10-GLM-4.7-Flash-Q6_K,22-lizavet,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.83,834,4096,
M10-GLM-4.7-Flash-Q6_K,23-tver-kuhni,1,ok,True,True,2,2,1,1,1,1,0.8,3,0,,1,67.98,769,3544,
M10-GLM-4.7-Flash-Q6_K,23-tver-kuhni,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.72,769,4096,
M10-GLM-4.7-Flash-Q6_K,23-tver-kuhni,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,78.73,769,4096,
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,79.04,774,4096,
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,2,ok,True,True,2,2,2,1,2,1,0.8,4,0,,1,47.36,774,2512,
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,3,ok,True,True,2,2,2,1,2,1,0.8,4,0,,1,47.27,774,2512,
M11-gemma-4-31B-it-Q6_K,01-ponchik,1,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,94.11,975,1480,
M12-T-pro-it-2.1-Q6_K,01-ponchik,1,format_fail,True,False,3,3,3,3,,1,0.9,5,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,26.11,927,383,
M12-T-pro-it-2.1-Q6_K,01-ponchik,2,format_fail,True,False,3,3,3,3,,1,0.9,5,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,22.67,927,383,
M12-T-pro-it-2.1-Q6_K,01-ponchik,3,format_fail,True,False,3,3,3,3,,1,0.9,5,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,22.67,927,383,
M12-T-pro-it-2.1-Q6_K,02-tabella-bakery,1,format_fail,True,False,2,2,2,2,,1,0.7,3,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,18.61,751,295,
M12-T-pro-it-2.1-Q6_K,02-tabella-bakery,2,format_fail,True,False,2,2,2,2,,1,0.7,3,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,17.09,751,295,
M12-T-pro-it-2.1-Q6_K,02-tabella-bakery,3,format_fail,True,False,2,2,2,2,,1,0.7,3,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,17.12,751,295,
M12-T-pro-it-2.1-Q6_K,03-surf-coffee,1,format_fail,True,False,3,3,3,3,,1,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,25.68,897,399,
M12-T-pro-it-2.1-Q6_K,03-surf-coffee,2,format_fail,True,False,3,3,3,3,,1,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,21.71,897,367,
M12-T-pro-it-2.1-Q6_K,03-surf-coffee,3,format_fail,True,False,3,3,3,3,,1,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,21.67,897,367,
M12-T-pro-it-2.1-Q6_K,04-buna-buna,1,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,17.41,777,273,
M12-T-pro-it-2.1-Q6_K,04-buna-buna,2,ok,True,True,2,2,2,1,1,1,0.85,5,1,,1,16.24,777,278,
M12-T-pro-it-2.1-Q6_K,04-buna-buna,3,ok,True,True,2,2,2,1,1,1,0.85,5,1,,1,16.18,777,278,
M12-T-pro-it-2.1-Q6_K,05-moncafe,1,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,19.3,816,303,
M12-T-pro-it-2.1-Q6_K,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,17.74,816,303,
M12-T-pro-it-2.1-Q6_K,05-moncafe,3,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,17.72,816,303,
M12-T-pro-it-2.1-Q6_K,06-breakfast-band,1,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,18.83,826,290,
M12-T-pro-it-2.1-Q6_K,06-breakfast-band,2,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,16.99,826,290,
M12-T-pro-it-2.1-Q6_K,06-breakfast-band,3,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,17.05,826,290,
M12-T-pro-it-2.1-Q6_K,07-ru-service,1,ok,True,True,1,1,1,1,1,1,0.9,3,0,,1,18.54,738,295,
M12-T-pro-it-2.1-Q6_K,07-ru-service,2,format_fail,True,False,1,1,1,1,,1,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,22.29,738,381,
M12-T-pro-it-2.1-Q6_K,07-ru-service,3,format_fail,True,False,1,1,1,1,,1,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,22.23,738,381,
M12-T-pro-it-2.1-Q6_K,08-avtobat,1,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,16.66,775,260,
M12-T-pro-it-2.1-Q6_K,08-avtobat,2,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,21.48,775,366,
M12-T-pro-it-2.1-Q6_K,08-avtobat,3,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,21.57,775,366,
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,1,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,24.4,1020,368,
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,2,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,22.12,1020,368,
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,3,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,22.15,1020,368,
M12-T-pro-it-2.1-Q6_K,10-krizanta,1,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,15.24,760,235,
M12-T-pro-it-2.1-Q6_K,10-krizanta,2,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,13.8,760,235,
M12-T-pro-it-2.1-Q6_K,10-krizanta,3,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,13.86,760,235,
M12-T-pro-it-2.1-Q6_K,11-u-nails,1,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,19.2,785,300,
M12-T-pro-it-2.1-Q6_K,11-u-nails,2,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,18.09,785,306,
M12-T-pro-it-2.1-Q6_K,11-u-nails,3,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,18.19,785,306,
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,18.95,742,299,
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,2,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,14.37,742,244,
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,3,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,14.41,742,244,
M12-T-pro-it-2.1-Q6_K,13-salon-radul,1,ok,True,True,2,2,2,1,1,1,0.9,6,0,,1,26.92,1008,409,
M12-T-pro-it-2.1-Q6_K,13-salon-radul,2,ok,True,True,2,2,2,1,1,1,0.9,6,0,,1,24.08,1008,401,
M12-T-pro-it-2.1-Q6_K,13-salon-radul,3,ok,True,True,2,2,2,1,1,1,0.9,6,0,,1,24.1,1008,401,
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,20.98,971,313,
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,2,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.26,971,322,
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,3,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.29,971,322,
M12-T-pro-it-2.1-Q6_K,15-cvety-bez-povoda,1,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,22.44,919,342,
M12-T-pro-it-2.1-Q6_K,15-cvety-bez-povoda,2,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,20.42,919,342,
M12-T-pro-it-2.1-Q6_K,15-cvety-bez-povoda,3,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,20.39,919,342,
M12-T-pro-it-2.1-Q6_K,16-kameliya,1,format_fail,True,False,3,3,3,3,,1,0.9,8,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,35.74,1099,546,
M12-T-pro-it-2.1-Q6_K,16-kameliya,2,format_fail,True,False,3,3,3,3,,1,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,28.07,1099,465,
M12-T-pro-it-2.1-Q6_K,16-kameliya,3,format_fail,True,False,3,3,3,3,,1,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,28.05,1099,465,
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,1,ok,True,True,2,2,2,1,1,1,0.9,7,1,,1,28.42,1110,427,
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,2,ok,True,True,2,2,2,1,1,1,0.9,7,1,,1,26.28,1110,435,
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,3,ok,True,True,2,2,2,1,1,1,0.9,7,1,,1,26.19,1110,435,
M12-T-pro-it-2.1-Q6_K,18-service69,1,ok,True,True,2,2,2,1,1,1,0.85,4,1,,1,20.87,947,313,
M12-T-pro-it-2.1-Q6_K,18-service69,2,ok,True,True,2,2,2,1,1,1,0.9,5,1,,1,22.13,947,369,
M12-T-pro-it-2.1-Q6_K,18-service69,3,ok,True,True,2,2,2,1,1,1,0.9,5,1,,1,22.12,947,369,
M12-T-pro-it-2.1-Q6_K,19-stecko-service,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,23.32,1101,343,
M12-T-pro-it-2.1-Q6_K,19-stecko-service,2,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,20.66,1101,343,
M12-T-pro-it-2.1-Q6_K,19-stecko-service,3,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,20.67,1101,343,
M12-T-pro-it-2.1-Q6_K,20-agym,1,ok,True,True,3,3,3,3,2,1,0.9,7,1,,1,26.28,1023,395,
M12-T-pro-it-2.1-Q6_K,20-agym,2,ok,True,True,3,3,3,3,2,1,0.9,7,1,,1,23.81,1023,395,
M12-T-pro-it-2.1-Q6_K,20-agym,3,ok,True,True,3,3,3,3,2,1,0.9,7,1,,1,23.76,1023,395,
M12-T-pro-it-2.1-Q6_K,21-my-fit,1,ok,True,True,3,3,3,3,1,1,0.9,5,1,,1,21.45,964,320,
M12-T-pro-it-2.1-Q6_K,21-my-fit,2,ok,True,True,3,3,3,3,1,1,0.9,5,1,,1,20.19,964,337,
M12-T-pro-it-2.1-Q6_K,21-my-fit,3,ok,True,True,3,3,3,3,1,1,0.9,5,1,,1,20.22,964,337,
M12-T-pro-it-2.1-Q6_K,22-lizavet,1,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,16.98,804,261,
M12-T-pro-it-2.1-Q6_K,22-lizavet,2,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.25,804,324,
M12-T-pro-it-2.1-Q6_K,22-lizavet,3,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.25,804,324,
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,1,ok,True,True,2,2,2,1,1,1,0.7,6,1,,1,23.88,757,372,
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,2,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,16.9,757,286,
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,3,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,16.93,757,286,
M12-T-pro-it-2.1-Q6_K,24-rustekstil,1,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,19.55,764,305,
M12-T-pro-it-2.1-Q6_K,24-rustekstil,2,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,18.89,764,318,
M12-T-pro-it-2.1-Q6_K,24-rustekstil,3,format_fail,True,False,2,2,2,1,,1,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено None",1,18.89,764,318,
M13-Qwen3.5-35B-A3B-UD-Q6_K,01-ponchik,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.93,977,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,01-ponchik,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.94,977,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,01-ponchik,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.89,977,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,02-tabella-bakery,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.31,794,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,02-tabella-bakery,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.03,794,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,02-tabella-bakery,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.78,794,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,03-surf-coffee,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.26,955,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,03-surf-coffee,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.06,955,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,03-surf-coffee,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.77,955,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,04-buna-buna,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.14,818,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,04-buna-buna,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.0,818,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,04-buna-buna,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.77,818,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,1,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,47.63,859,3136,
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,47.52,859,3136,
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,3,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,47.26,859,3136,
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,1,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,40.86,861,2685,
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,2,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,40.72,861,2685,
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,3,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,40.5,861,2685,
M13-Qwen3.5-35B-A3B-UD-Q6_K,07-ru-service,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.1,785,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,07-ru-service,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.99,785,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,07-ru-service,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.79,785,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,08-avtobat,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.13,812,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,08-avtobat,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.03,812,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,08-avtobat,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.8,812,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,09-geometriya-koles,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.42,1064,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,09-geometriya-koles,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.09,1064,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,09-geometriya-koles,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.82,1064,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,10-krizanta,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.11,798,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,10-krizanta,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.05,798,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,10-krizanta,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.77,798,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,11-u-nails,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.13,823,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,11-u-nails,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.04,823,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,11-u-nails,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.79,823,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,12-parikmaherskaya-volokolamskaya,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.13,788,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,12-parikmaherskaya-volokolamskaya,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.04,788,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,12-parikmaherskaya-volokolamskaya,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.79,788,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,13-salon-radul,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.39,1053,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,13-salon-radul,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.04,1053,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,13-salon-radul,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.83,1053,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,14-milliard-roz,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.35,1008,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,14-milliard-roz,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.11,1008,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,14-milliard-roz,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.83,1008,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,15-cvety-bez-povoda,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.23,963,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,15-cvety-bez-povoda,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.1,963,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,15-cvety-bez-povoda,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.81,963,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,16-kameliya,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.51,1142,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,16-kameliya,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.13,1142,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,16-kameliya,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.84,1142,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,17-ip-electronics,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.48,1137,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,17-ip-electronics,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.13,1137,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,17-ip-electronics,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.85,1137,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,18-service69,1,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,62.33,990,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,18-service69,2,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,62.08,990,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,18-service69,3,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,61.81,990,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,19-stecko-service,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.49,1138,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,19-stecko-service,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.11,1138,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,19-stecko-service,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.86,1138,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,20-agym,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.39,1068,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,20-agym,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.07,1068,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,20-agym,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.84,1068,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,21-my-fit,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.38,1028,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,21-my-fit,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.09,1028,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,21-my-fit,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.84,1028,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,22-lizavet,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.16,852,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,22-lizavet,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.05,852,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,22-lizavet,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.77,852,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,23-tver-kuhni,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.14,798,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,23-tver-kuhni,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.02,798,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,23-tver-kuhni,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.77,798,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,24-rustekstil,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.12,806,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,24-rustekstil,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,62.06,806,4096,
M13-Qwen3.5-35B-A3B-UD-Q6_K,24-rustekstil,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,61.79,806,4096,
M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,1,format_fail,True,False,3,4,3,4,1,0,1,7,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.42,1992,535,
M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,2,format_fail,True,False,3,4,3,4,1,0,1,7,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.18,1992,478,
M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,3,format_fail,True,False,3,4,3,4,1,0,1,7,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.18,1992,478,
M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,1,format_fail,True,False,2,3,2,3,0,0,1,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.55,1819,372,
M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,2,format_fail,True,False,2,2,1,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.54,1819,290,
M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,3,format_fail,True,False,2,2,1,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.54,1819,290,
M6-GigaChat3.1-10B-A1.8B-q8_0,03-surf-coffee,1,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,3.3,1973,331,
M6-GigaChat3.1-10B-A1.8B-q8_0,03-surf-coffee,2,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.79,1973,319,
M6-GigaChat3.1-10B-A1.8B-q8_0,03-surf-coffee,3,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.79,1973,319,
M6-GigaChat3.1-10B-A1.8B-q8_0,04-buna-buna,1,format_fail,True,False,2,2,2,2,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.08,1848,312,
M6-GigaChat3.1-10B-A1.8B-q8_0,04-buna-buna,2,format_fail,True,False,2,3,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.08,1848,352,
M6-GigaChat3.1-10B-A1.8B-q8_0,04-buna-buna,3,format_fail,True,False,2,3,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.08,1848,352,
M6-GigaChat3.1-10B-A1.8B-q8_0,05-moncafe,1,format_fail,True,False,2,2,2,2,0,0,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.45,1882,353,
M6-GigaChat3.1-10B-A1.8B-q8_0,05-moncafe,2,format_fail,True,False,2,2,2,2,0,0,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,3.24,1882,371,
M6-GigaChat3.1-10B-A1.8B-q8_0,05-moncafe,3,format_fail,True,False,2,2,2,2,0,0,0.9,6,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,3.24,1882,371,
M6-GigaChat3.1-10B-A1.8B-q8_0,06-breakfast-band,1,format_fail,True,False,2,3,2,2,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.29,1900,334,
M6-GigaChat3.1-10B-A1.8B-q8_0,06-breakfast-band,2,format_fail,True,False,2,3,2,2,0,0,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.03,1900,347,
M6-GigaChat3.1-10B-A1.8B-q8_0,06-breakfast-band,3,format_fail,True,False,2,3,2,2,0,0,0.8,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.03,1900,347,
M6-GigaChat3.1-10B-A1.8B-q8_0,07-ru-service,1,format_fail,True,False,2,3,2,2,0,0,0.9,5,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.03,1814,427,
M6-GigaChat3.1-10B-A1.8B-q8_0,07-ru-service,2,format_fail,True,False,2,2,1,1,0,0,1,4,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.91,1814,333,
M6-GigaChat3.1-10B-A1.8B-q8_0,07-ru-service,3,format_fail,True,False,2,2,1,1,0,0,1,4,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.91,1814,333,
M6-GigaChat3.1-10B-A1.8B-q8_0,08-avtobat,1,format_fail,True,False,2,3,2,2,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.39,1844,349,
M6-GigaChat3.1-10B-A1.8B-q8_0,08-avtobat,2,format_fail,True,False,2,3,2,2,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.12,1844,357,
M6-GigaChat3.1-10B-A1.8B-q8_0,08-avtobat,3,format_fail,True,False,2,3,2,2,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.12,1844,357,
M6-GigaChat3.1-10B-A1.8B-q8_0,09-geometriya-koles,1,format_fail,True,False,3,4,4,4,1,0,1,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.75,2073,366,
M6-GigaChat3.1-10B-A1.8B-q8_0,09-geometriya-koles,2,format_fail,True,False,3,4,4,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.69,2073,422,
M6-GigaChat3.1-10B-A1.8B-q8_0,09-geometriya-koles,3,format_fail,True,False,3,4,4,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.69,2073,422,
M6-GigaChat3.1-10B-A1.8B-q8_0,10-krizanta,1,format_fail,True,False,2,3,2,2,0,0,1,4,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.8,1828,281,
M6-GigaChat3.1-10B-A1.8B-q8_0,10-krizanta,2,format_fail,True,False,2,3,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,2.25,1828,257,
M6-GigaChat3.1-10B-A1.8B-q8_0,10-krizanta,3,format_fail,True,False,2,3,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,2.25,1828,257,
M6-GigaChat3.1-10B-A1.8B-q8_0,11-u-nails,1,format_fail,True,False,2,2,2,2,0,0,0.9,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.68,1854,382,
M6-GigaChat3.1-10B-A1.8B-q8_0,11-u-nails,2,format_fail,True,False,2,2,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.63,1854,301,
M6-GigaChat3.1-10B-A1.8B-q8_0,11-u-nails,3,format_fail,True,False,2,2,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.63,1854,301,
M6-GigaChat3.1-10B-A1.8B-q8_0,12-parikmaherskaya-volokolamskaya,1,format_fail,True,False,2,2,2,2,0,0,1,6,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.53,1817,369,
M6-GigaChat3.1-10B-A1.8B-q8_0,12-parikmaherskaya-volokolamskaya,2,format_fail,True,False,2,2,2,2,0,0,1,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.86,1817,327,
M6-GigaChat3.1-10B-A1.8B-q8_0,12-parikmaherskaya-volokolamskaya,3,format_fail,True,False,2,2,2,2,0,0,1,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.86,1817,327,
M6-GigaChat3.1-10B-A1.8B-q8_0,13-salon-radul,1,format_fail,True,False,2,4,2,2,0,0,1,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.31,2082,432,
M6-GigaChat3.1-10B-A1.8B-q8_0,13-salon-radul,2,format_fail,True,False,2,4,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.63,2082,415,
M6-GigaChat3.1-10B-A1.8B-q8_0,13-salon-radul,3,format_fail,True,False,2,4,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.62,2082,415,
M6-GigaChat3.1-10B-A1.8B-q8_0,14-milliard-roz,1,format_fail,True,False,2,2,2,1,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.98,2041,395,
M6-GigaChat3.1-10B-A1.8B-q8_0,14-milliard-roz,2,format_fail,True,False,2,2,2,1,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.69,2041,422,
M6-GigaChat3.1-10B-A1.8B-q8_0,14-milliard-roz,3,format_fail,True,False,2,2,2,1,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.69,2041,422,
M6-GigaChat3.1-10B-A1.8B-q8_0,15-cvety-bez-povoda,1,format_fail,True,False,2,2,1,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.84,1988,390,
M6-GigaChat3.1-10B-A1.8B-q8_0,15-cvety-bez-povoda,2,format_fail,True,False,2,2,1,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.05,1988,348,
M6-GigaChat3.1-10B-A1.8B-q8_0,15-cvety-bez-povoda,3,format_fail,True,False,2,2,1,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.05,1988,348,
M6-GigaChat3.1-10B-A1.8B-q8_0,16-kameliya,1,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.17,2172,518,
M6-GigaChat3.1-10B-A1.8B-q8_0,16-kameliya,2,format_fail,True,False,3,4,3,4,1,0,1,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,5.87,2172,669,
M6-GigaChat3.1-10B-A1.8B-q8_0,16-kameliya,3,format_fail,True,False,3,4,3,4,1,0,1,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,5.86,2172,669,
M6-GigaChat3.1-10B-A1.8B-q8_0,17-ip-electronics,1,format_fail,True,False,3,4,2,2,0,0,1,8,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,4.22,2168,409,
M6-GigaChat3.1-10B-A1.8B-q8_0,17-ip-electronics,2,format_fail,True,False,3,4,2,2,0,0,0.9,8,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.1,2168,467,
M6-GigaChat3.1-10B-A1.8B-q8_0,17-ip-electronics,3,format_fail,True,False,3,4,2,2,0,0,0.9,8,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.1,2168,467,
M6-GigaChat3.1-10B-A1.8B-q8_0,18-service69,1,format_fail,True,False,2,3,2,1,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.59,2020,357,
M6-GigaChat3.1-10B-A1.8B-q8_0,18-service69,2,format_fail,True,False,2,3,2,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.65,2020,301,
M6-GigaChat3.1-10B-A1.8B-q8_0,18-service69,3,format_fail,True,False,2,3,2,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.66,2020,301,
M6-GigaChat3.1-10B-A1.8B-q8_0,19-stecko-service,1,format_fail,True,False,2,3,2,2,0,0,1,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.99,2152,388,
M6-GigaChat3.1-10B-A1.8B-q8_0,19-stecko-service,2,format_fail,True,False,2,3,2,2,0,0,1,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.78,2152,431,
M6-GigaChat3.1-10B-A1.8B-q8_0,19-stecko-service,3,format_fail,True,False,2,3,2,2,0,0,1,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.78,2152,431,
M6-GigaChat3.1-10B-A1.8B-q8_0,20-agym,1,format_fail,True,False,3,4,3,4,2,0,1,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.73,2088,364,
M6-GigaChat3.1-10B-A1.8B-q8_0,20-agym,2,format_fail,True,False,3,4,4,4,2,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.91,2088,447,
M6-GigaChat3.1-10B-A1.8B-q8_0,20-agym,3,format_fail,True,False,3,4,4,4,2,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.9,2088,447,
M6-GigaChat3.1-10B-A1.8B-q8_0,21-my-fit,1,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.73,2046,367,
M6-GigaChat3.1-10B-A1.8B-q8_0,21-my-fit,2,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.61,2046,412,
M6-GigaChat3.1-10B-A1.8B-q8_0,21-my-fit,3,format_fail,True,False,3,4,3,4,1,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.61,2046,412,
M6-GigaChat3.1-10B-A1.8B-q8_0,22-lizavet,1,format_fail,True,False,2,3,2,2,0,0,0.9,5,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.28,1880,334,
M6-GigaChat3.1-10B-A1.8B-q8_0,22-lizavet,2,format_fail,True,False,2,3,2,2,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.23,1880,370,
M6-GigaChat3.1-10B-A1.8B-q8_0,22-lizavet,3,format_fail,True,False,2,3,2,2,0,0,0.9,4,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.23,1880,370,
M6-GigaChat3.1-10B-A1.8B-q8_0,23-tver-kuhni,1,format_fail,True,False,2,3,2,2,0,0,1,3,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.28,1831,337,
M6-GigaChat3.1-10B-A1.8B-q8_0,23-tver-kuhni,2,format_fail,True,False,2,3,2,2,0,0,0.9,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.11,1831,356,
M6-GigaChat3.1-10B-A1.8B-q8_0,23-tver-kuhni,3,format_fail,True,False,2,3,2,2,0,0,0.9,5,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.11,1831,356,
M6-GigaChat3.1-10B-A1.8B-q8_0,24-rustekstil,1,format_fail,True,False,2,2,1,1,0,0,1,3,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,3.2,1837,327,
M6-GigaChat3.1-10B-A1.8B-q8_0,24-rustekstil,2,format_fail,True,False,2,2,1,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.91,1837,333,
M6-GigaChat3.1-10B-A1.8B-q8_0,24-rustekstil,3,format_fail,True,False,2,2,1,1,0,0,1,3,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,2.91,1837,333,
M7-gpt-oss-20b-Q8_0,01-ponchik,1,ok,True,True,3,2,2,3,2,1,0.7,5,1,,1,29.57,991,2479,
M7-gpt-oss-20b-Q8_0,01-ponchik,2,ok,True,True,3,3,2,3,2,1,0.75,6,1,,1,19.99,991,1750,
M7-gpt-oss-20b-Q8_0,01-ponchik,3,ok,True,True,3,2,2,3,2,1,0.9,5,1,,1,17.45,991,1545,
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,1,ok,True,True,2,2,1,2,1,1,0.5,2,0,,1,15.34,829,1311,
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,2,ok,True,True,1,2,1,2,1,1,0.6,5,1,,1,22.08,829,1946,
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,3,ok,True,True,1,2,1,1,1,1,0.45,3,1,,1,17.72,829,1555,
M7-gpt-oss-20b-Q8_0,03-surf-coffee,1,ok,True,True,1,2,2,1,1,1,0.4,4,1,,1,21.04,965,1787,
M7-gpt-oss-20b-Q8_0,03-surf-coffee,2,ok,True,True,2,5,2,3,1,1,0.7,3,1,,1,22.38,965,1956,
M7-gpt-oss-20b-Q8_0,03-surf-coffee,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,47.33,965,4096,
M7-gpt-oss-20b-Q8_0,04-buna-buna,1,ok,True,True,2,2,2,1,1,1,0.7,4,0,,1,27.79,838,2381,
M7-gpt-oss-20b-Q8_0,04-buna-buna,2,ok,True,True,1,2,2,1,1,1,0.5,4,0,,1,21.22,838,1859,
M7-gpt-oss-20b-Q8_0,04-buna-buna,3,ok,True,True,2,2,2,1,1,1,0.6,2,1,,1,11.63,838,1022,
M7-gpt-oss-20b-Q8_0,05-moncafe,1,ok,True,True,1,1,1,1,1,1,0.8,5,1,,1,10.38,882,859,
M7-gpt-oss-20b-Q8_0,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.6,5,0,,1,20.13,882,1770,
M7-gpt-oss-20b-Q8_0,05-moncafe,3,ok,True,True,2,2,2,1,1,1,0.6,3,1,,1,22.05,882,1935,
M7-gpt-oss-20b-Q8_0,06-breakfast-band,1,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,20.63,882,1758,
M7-gpt-oss-20b-Q8_0,06-breakfast-band,2,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,10.47,882,918,
M7-gpt-oss-20b-Q8_0,06-breakfast-band,3,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,23.87,882,2090,
M7-gpt-oss-20b-Q8_0,07-ru-service,1,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,11.03,807,921,
M7-gpt-oss-20b-Q8_0,07-ru-service,2,ok,True,True,1,1,1,1,1,1,0.9,5,1,,1,18.56,807,1628,
M7-gpt-oss-20b-Q8_0,07-ru-service,3,ok,True,True,1,2,1,1,1,1,0.7,4,0,,1,17.4,807,1529,
M7-gpt-oss-20b-Q8_0,08-avtobat,1,ok,True,True,2,2,1,1,1,1,0.4,4,1,,1,14.45,834,1224,
M7-gpt-oss-20b-Q8_0,08-avtobat,2,ok,True,True,2,2,1,1,1,1,0.6,3,1,,1,8.44,834,737,
M7-gpt-oss-20b-Q8_0,08-avtobat,3,ok,True,True,2,2,1,2,1,1,0.6,4,1,,1,8.97,834,796,
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,1,ok,True,True,2,2,2,2,1,1,0.9,3,1,,1,15.62,1052,1307,
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,2,ok,True,True,2,5,3,2,1,1,0.7,5,1,,1,10.68,1052,935,
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,3,ok,True,True,2,3,2,3,1,1,0.75,3,1,,1,40.69,1052,3540,
M7-gpt-oss-20b-Q8_0,10-krizanta,1,ok,True,True,2,2,1,1,1,1,0.6,3,1,,1,13.15,829,1105,
M7-gpt-oss-20b-Q8_0,10-krizanta,2,ok,True,True,2,2,1,2,1,1,0.55,5,1,,1,38.94,829,3387,
M7-gpt-oss-20b-Q8_0,10-krizanta,3,ok,True,True,2,2,1,1,1,1,0.4,4,1,,1,20.98,829,1840,
M7-gpt-oss-20b-Q8_0,11-u-nails,1,ok,True,True,1,2,2,1,1,1,0.75,5,1,,1,21.63,849,1849,
M7-gpt-oss-20b-Q8_0,11-u-nails,2,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,32.28,849,2818,
M7-gpt-oss-20b-Q8_0,11-u-nails,3,ok,True,True,2,2,2,2,1,1,0.6,5,1,,1,18.21,849,1599,
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,1,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,14.63,811,1238,
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,2,ok,True,True,2,2,1,1,1,1,0.6,5,1,,1,23.01,811,2017,
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,3,ok,True,True,2,2,1,2,1,1,0.4,2,0,,1,16.98,811,1492,
M7-gpt-oss-20b-Q8_0,13-salon-radul,1,ok,True,True,2,2,2,1,1,1,0.7,5,1,,1,32.64,1089,2783,
M7-gpt-oss-20b-Q8_0,13-salon-radul,2,ok,True,True,2,3,2,1,2,1,0.85,7,1,,1,27.85,1089,2447,
M7-gpt-oss-20b-Q8_0,13-salon-radul,3,ok,True,True,2,2,2,1,1,1,0.73,5,1,,1,11.19,1089,979,
M7-gpt-oss-20b-Q8_0,14-milliard-roz,1,ok,True,True,1,1,1,1,1,1,0.92,5,1,,1,15.16,1033,1269,
M7-gpt-oss-20b-Q8_0,14-milliard-roz,2,ok,True,True,1,2,1,1,1,1,0.7,5,1,,1,32.93,1033,2864,
M7-gpt-oss-20b-Q8_0,14-milliard-roz,3,ok,True,True,2,2,1,1,1,1,0.6,5,0,,1,18.85,1033,1649,
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,1,ok,True,True,1,2,1,1,1,1,0.68,5,1,,1,16.68,1005,1403,
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,2,ok,True,True,1,2,1,1,1,1,0.7,5,1,,1,17.08,1005,1498,
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,3,ok,True,True,1,2,1,1,1,1,0.45,1,1,,1,31.35,1005,2736,
M7-gpt-oss-20b-Q8_0,16-kameliya,1,ok,True,True,2,2,2,2,2,1,0.8,6,1,,1,14.37,1160,1188,
M7-gpt-oss-20b-Q8_0,16-kameliya,2,ok,True,True,3,3,2,3,2,1,0.8,6,1,,1,25.91,1160,2260,
M7-gpt-oss-20b-Q8_0,16-kameliya,3,ok,True,True,3,3,2,3,1,1,0.8,7,1,,1,25.19,1160,2206,
M7-gpt-oss-20b-Q8_0,17-ip-electronics,1,ok,True,True,2,3,2,1,1,1,0.85,5,1,,1,11.27,1148,928,
M7-gpt-oss-20b-Q8_0,17-ip-electronics,2,ok,True,True,2,2,1,1,1,1,0.55,6,1,,1,13.56,1148,1192,
M7-gpt-oss-20b-Q8_0,17-ip-electronics,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,46.96,1148,4096,
M7-gpt-oss-20b-Q8_0,18-service69,1,ok,True,True,1,2,1,1,1,1,0.85,3,0,,1,31.08,995,2671,
M7-gpt-oss-20b-Q8_0,18-service69,2,ok,True,True,1,2,1,1,1,1,0.68,4,1,,1,24.79,995,2158,
M7-gpt-oss-20b-Q8_0,18-service69,3,ok,True,True,2,2,1,1,1,1,0.4,3,0,,1,18.05,995,1581,
M7-gpt-oss-20b-Q8_0,19-stecko-service,1,ok,True,True,1,2,1,2,1,1,0.85,3,1,,1,27.15,1136,2301,
M7-gpt-oss-20b-Q8_0,19-stecko-service,2,ok,True,True,2,2,2,1,1,1,0.7,4,1,,1,22.14,1136,1933,
M7-gpt-oss-20b-Q8_0,19-stecko-service,3,ok,True,True,1,2,1,1,1,1,0.7,3,1,,1,11.3,1136,987,
M7-gpt-oss-20b-Q8_0,20-agym,1,ok,True,True,3,3,2,2,2,1,0.7,5,1,,1,21.86,1078,1851,
M7-gpt-oss-20b-Q8_0,20-agym,2,ok,True,True,2,2,2,2,2,1,0.92,6,1,,1,24.15,1078,2108,
M7-gpt-oss-20b-Q8_0,20-agym,3,format_fail,True,False,3,3,3,3,2,0,0.7,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0; evidence[4].quote: пуст или не строка",1,10.82,1078,948,
M7-gpt-oss-20b-Q8_0,21-my-fit,1,ok,True,True,2,2,2,1,1,1,0.7,5,1,,1,13.62,1058,1134,
M7-gpt-oss-20b-Q8_0,21-my-fit,2,ok,True,True,2,2,2,2,1,1,0.7,4,1,,1,37.47,1058,3256,
M7-gpt-oss-20b-Q8_0,21-my-fit,3,ok,True,True,2,2,2,1,1,1,0.7,4,1,,1,12.05,1058,1050,
M7-gpt-oss-20b-Q8_0,22-lizavet,1,ok,True,True,2,2,2,1,1,1,0.75,3,1,,1,12.31,884,1030,
M7-gpt-oss-20b-Q8_0,22-lizavet,2,ok,True,True,1,2,1,1,1,1,0.4,5,1,,1,37.37,884,3255,
M7-gpt-oss-20b-Q8_0,22-lizavet,3,ok,True,True,2,2,2,2,1,1,0.65,3,1,,1,20.16,884,1768,
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,1,ok,True,True,1,1,1,1,1,1,0.6,3,1,,1,12.65,821,1065,
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,2,ok,True,True,1,1,1,1,1,1,0.6,3,1,,1,6.16,821,533,
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,3,ok,True,True,1,1,1,1,1,1,0.45,4,1,,1,20.93,821,1848,
M7-gpt-oss-20b-Q8_0,24-rustekstil,1,ok,True,True,1,1,1,1,1,1,0.9,5,1,,1,31.67,827,2722,
M7-gpt-oss-20b-Q8_0,24-rustekstil,2,ok,True,True,1,2,1,1,1,1,0.5,3,1,,1,16.31,827,1430,
M7-gpt-oss-20b-Q8_0,24-rustekstil,3,ok,True,True,2,2,1,1,1,1,0.45,2,1,,1,19.55,827,1720,
M8-saiga_gemma3_12b-Q8_0,01-ponchik,1,format_fail,True,False,3,3,3,4,3,0,1,7,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,11.97,968,405,
M8-saiga_gemma3_12b-Q8_0,01-ponchik,2,format_fail,True,False,3,3,3,4,2,0,1,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,10.43,968,388,
M8-saiga_gemma3_12b-Q8_0,01-ponchik,3,format_fail,True,False,3,4,2,4,3,0,0.9,6,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.97,968,331,
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,1,ok,True,True,2,2,2,2,1,1,0.6,5,1,,1,10.32,781,336,
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,2,ok,True,True,2,3,1,2,1,1,0.75,7,1,,1,10.78,781,406,
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,3,ok,True,True,3,3,2,3,2,1,0.8,5,1,,1,9.71,781,364,
M8-saiga_gemma3_12b-Q8_0,03-surf-coffee,1,format_fail,True,False,3,4,3,4,3,0,0.9,8,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,11.49,944,373,
M8-saiga_gemma3_12b-Q8_0,03-surf-coffee,2,format_fail,True,False,4,4,3,3,4,0,1,7,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,9.99,944,372,
M8-saiga_gemma3_12b-Q8_0,03-surf-coffee,3,format_fail,True,False,3,4,2,3,1,0,0.9,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,10.24,944,382,
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,1,format_fail,True,False,3,4,2,3,2,0,0.8,7,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,11.46,806,381,
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,2,ok,True,True,2,4,2,2,1,1,0.8,4,1,,1,8.81,806,327,
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,3,format_fail,True,False,2,4,2,2,1,0,0.9,4,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.53,806,316,
M8-saiga_gemma3_12b-Q8_0,05-moncafe,1,ok,True,True,3,3,3,3,2,1,1,6,1,,1,10.75,849,350,
M8-saiga_gemma3_12b-Q8_0,05-moncafe,2,format_fail,True,False,2,2,2,1,1,0,0.8,4,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.57,849,278,
M8-saiga_gemma3_12b-Q8_0,05-moncafe,3,format_fail,True,False,2,4,1,1,0,0,0.6,7,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,9.83,849,367,
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,1,ok,True,True,2,2,2,1,1,1,0.8,6,1,,1,10.48,851,340,
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,2,format_fail,True,False,2,3,1,2,1,0,0.7,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.03,851,296,
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,3,ok,True,True,2,4,1,2,1,1,0.7,5,1,,1,10.45,851,392,
M8-saiga_gemma3_12b-Q8_0,07-ru-service,1,ok,True,True,1,2,1,1,1,1,0.6,3,1,,1,8.55,773,270,
M8-saiga_gemma3_12b-Q8_0,07-ru-service,2,ok,True,True,2,3,2,1,1,1,0.8,5,1,,1,8.76,773,326,
M8-saiga_gemma3_12b-Q8_0,07-ru-service,3,ok,True,True,1,2,1,1,1,1,0.6,6,1,,1,8.97,773,335,
M8-saiga_gemma3_12b-Q8_0,08-avtobat,1,ok,True,True,2,3,2,1,1,1,0.7,4,1,,1,9.96,802,321,
M8-saiga_gemma3_12b-Q8_0,08-avtobat,2,format_fail,True,False,3,3,2,3,0,0,0.7,8,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,11.56,802,436,
M8-saiga_gemma3_12b-Q8_0,08-avtobat,3,format_fail,True,False,2,3,2,1,1,0,0.8,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,9.14,802,341,
M8-saiga_gemma3_12b-Q8_0,09-geometriya-koles,1,ok,True,True,3,4,3,3,3,1,1,6,1,,1,12.97,1051,427,
M9-gemma-4-12b-it-Q8_0,01-ponchik,1,ok,True,True,3,4,3,4,2,1,0.95,5,1,,1,52.71,975,1944,
M9-gemma-4-12b-it-Q8_0,01-ponchik,2,ok,True,True,3,2,3,3,2,1,0.9,6,0,,1,85.01,975,3214,
M9-gemma-4-12b-it-Q8_0,01-ponchik,3,ok,True,True,3,3,3,3,3,1,0.9,6,1,,1,88.09,975,3334,
M9-gemma-4-12b-it-Q8_0,02-tabella-bakery,1,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,109.47,788,4096,
M9-gemma-4-12b-it-Q8_0,02-tabella-bakery,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,108.18,788,4096,
M9-gemma-4-12b-it-Q8_0,02-tabella-bakery,3,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,108.2,788,4096,
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,1,ok,True,True,4,4,3,4,4,1,0.9,5,0,,1,51.41,951,1881,
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,2,ok,True,True,4,4,3,4,4,1,0.85,6,0,,1,51.99,951,1961,
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,3,ok,True,True,3,3,3,3,3,1,0.9,5,0,,1,66.54,951,2516,
M9-gemma-4-12b-it-Q8_0,04-buna-buna,1,ok,True,True,2,2,2,1,1,1,0.85,3,0,,1,45.94,813,1682,
M9-gemma-4-12b-it-Q8_0,04-buna-buna,2,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,39.17,813,1475,
M9-gemma-4-12b-it-Q8_0,04-buna-buna,3,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,42.63,813,1606,
M9-gemma-4-12b-it-Q8_0,05-moncafe,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,40.86,856,1485,
M9-gemma-4-12b-it-Q8_0,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.7,4,1,,1,65.14,856,2464,
M9-gemma-4-12b-it-Q8_0,05-moncafe,3,ok,True,True,2,2,1,1,1,1,0.9,4,0,,1,43.46,856,1636,
M9-gemma-4-12b-it-Q8_0,06-breakfast-band,1,ok,True,True,2,2,2,1,1,1,0.7,3,0,,1,60.03,858,2216,
gemma-7b-it.Q8_0.gguf,01-ponchik,1,format_fail,True,False,3,2,2,2,1,0,0.8,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.58,982,337,
gemma-7b-it.Q8_0.gguf,01-ponchik,2,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,6.25,982,371,
gemma-7b-it.Q8_0.gguf,01-ponchik,3,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,4.85,982,287,
gemma-7b-it.Q8_0.gguf,02-tabella-bakery,1,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,11.77,793,665,
gemma-7b-it.Q8_0.gguf,02-tabella-bakery,2,format_fail,True,False,2,1,2,0,0,0,0.8,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.47,793,326,
gemma-7b-it.Q8_0.gguf,02-tabella-bakery,3,format_fail,True,False,2,1,2,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.4,793,322,
gemma-7b-it.Q8_0.gguf,03-surf-coffee,1,format_fail,True,False,2,2,1,1,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.3,959,391,
gemma-7b-it.Q8_0.gguf,03-surf-coffee,2,format_fail,True,False,2,2,1,1,0,0,0.8,1,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.63,959,334,
gemma-7b-it.Q8_0.gguf,03-surf-coffee,3,format_fail,True,False,2,2,2,1,1,0,0.8,1,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.86,959,288,
gemma-7b-it.Q8_0.gguf,04-buna-buna,1,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.86,820,373,
gemma-7b-it.Q8_0.gguf,04-buna-buna,2,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.19,820,308,
gemma-7b-it.Q8_0.gguf,04-buna-buna,3,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.72,820,280,
gemma-7b-it.Q8_0.gguf,05-moncafe,1,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.5,862,347,
gemma-7b-it.Q8_0.gguf,05-moncafe,2,format_fail,True,False,2,1,2,0,0,0,0.5,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.6,862,392,
gemma-7b-it.Q8_0.gguf,05-moncafe,3,format_fail,True,False,2,2,1,0,0,0,0.5,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.06,862,361,
gemma-7b-it.Q8_0.gguf,06-breakfast-band,1,format_fail,True,False,1,1,0,0,0,0,0,1,0,"criteria.process_automation: ожидалось целое 1–5, получено 0; criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.97,863,312,
gemma-7b-it.Q8_0.gguf,06-breakfast-band,2,format_fail,True,False,2,2,1,0,0,0,0.6,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.69,863,455,
gemma-7b-it.Q8_0.gguf,06-breakfast-band,3,format_fail,True,False,2,2,1,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.96,863,413,
gemma-7b-it.Q8_0.gguf,07-ru-service,1,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.11,783,327,
gemma-7b-it.Q8_0.gguf,07-ru-service,2,format_fail,True,False,2,1,2,0,0,0,0.5,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.49,783,386,
gemma-7b-it.Q8_0.gguf,07-ru-service,3,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.55,783,331,
gemma-7b-it.Q8_0.gguf,08-avtobat,1,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,10.31,814,577,
gemma-7b-it.Q8_0.gguf,08-avtobat,2,format_fail,True,False,,,,,,,,0,0,"level: ожидалось целое 1–5, получено None; criteria: ожидался объект; criteria.online_presence: ожидалось целое 1–5, получено None; criteria.process_automation: ожидалось целое 1–5, получено None; criteria.ecommerce_online_services: ожидалось целое 1–5, получено None; criteria.data_driven: ожидалось целое 1–5, получено None; criteria.ai_usage: ожидалось целое 1–5, получено None; evidence: ожидался массив; confidence: ожидалось число 0–1, получено None; level_rationale: пуст или не строка",1,11.35,814,675,
gemma-7b-it.Q8_0.gguf,08-avtobat,3,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.49,814,326,
gemma-7b-it.Q8_0.gguf,09-geometriya-koles,1,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.82,1073,237,
gemma-7b-it.Q8_0.gguf,09-geometriya-koles,2,format_fail,True,False,2,2,1,0,0,0,0.8,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.4,1073,320,
gemma-7b-it.Q8_0.gguf,09-geometriya-koles,3,format_fail,True,False,2,3,2,2,1,0,0.8,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.81,1073,344,
gemma-7b-it.Q8_0.gguf,10-krizanta,1,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.98,802,259,
gemma-7b-it.Q8_0.gguf,10-krizanta,2,format_fail,True,False,2,1,2,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.41,802,380,
gemma-7b-it.Q8_0.gguf,10-krizanta,3,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,9.93,802,590,
gemma-7b-it.Q8_0.gguf,11-u-nails,1,format_fail,True,False,2,2,1,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,4.73,825,244,
gemma-7b-it.Q8_0.gguf,11-u-nails,2,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.37,825,320,
gemma-7b-it.Q8_0.gguf,11-u-nails,3,format_fail,True,False,2,2,1,0,0,0,0.75,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.03,825,299,
gemma-7b-it.Q8_0.gguf,12-parikmaherskaya-volokolamskaya,1,format_fail,True,False,2,2,1,0,0,0,0.5,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.95,786,377,
gemma-7b-it.Q8_0.gguf,12-parikmaherskaya-volokolamskaya,2,format_fail,True,False,2,1,2,0,0,0,0,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.03,786,298,
gemma-7b-it.Q8_0.gguf,12-parikmaherskaya-volokolamskaya,3,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,9.52,786,566,
gemma-7b-it.Q8_0.gguf,13-salon-radul,1,format_fail,True,False,3,2,1,1,0,0,0.5,1,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.5,1065,457,
gemma-7b-it.Q8_0.gguf,13-salon-radul,2,format_fail,True,False,3,2,1,1,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.02,1065,297,
gemma-7b-it.Q8_0.gguf,13-salon-radul,3,format_fail,True,False,2,2,1,0,0,0,0.7,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.53,1065,387,
gemma-7b-it.Q8_0.gguf,14-milliard-roz,1,format_fail,True,False,2,1,1,0,0,0,0.5,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.45,1025,395,
gemma-7b-it.Q8_0.gguf,14-milliard-roz,2,format_fail,True,False,2,2,1,1,1,0,0.8,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.38,1025,319,
gemma-7b-it.Q8_0.gguf,14-milliard-roz,3,format_fail,True,False,2,2,1,1,0,0,0.8,1,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.36,1025,316,
gemma-7b-it.Q8_0.gguf,15-cvety-bez-povoda,1,format_fail,True,False,2,2,1,0,0,0,0.7,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.71,983,297,
gemma-7b-it.Q8_0.gguf,15-cvety-bez-povoda,2,format_fail,True,False,2,2,1,1,0,0,0.8,1,0,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.35,983,317,
gemma-7b-it.Q8_0.gguf,15-cvety-bez-povoda,3,format_fail,True,False,2,1,2,0,0,0,0.5,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.4,983,379,
gemma-7b-it.Q8_0.gguf,16-kameliya,1,format_fail,True,False,3,2,1,2,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.97,1147,351,
gemma-7b-it.Q8_0.gguf,16-kameliya,2,format_fail,True,False,2,2,1,1,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.18,1147,366,
gemma-7b-it.Q8_0.gguf,16-kameliya,3,format_fail,True,False,3,3,2,2,1,0,0.8,1,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.41,1147,497,
gemma-7b-it.Q8_0.gguf,17-ip-electronics,1,format_fail,True,False,3,2,1,1,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.1,1158,305,
gemma-7b-it.Q8_0.gguf,17-ip-electronics,2,no_json,False,False,,,,,,,,,0,JSON не найден в ответе,1,5.36,1158,317,
gemma-7b-it.Q8_0.gguf,17-ip-electronics,3,format_fail,True,False,2,2,1,0,0,0,0.8,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.15,1158,482,
gemma-7b-it.Q8_0.gguf,18-service69,1,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.45,995,338,
gemma-7b-it.Q8_0.gguf,18-service69,2,format_fail,True,False,2,2,1,0,0,0,0.5,2,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.56,995,388,
gemma-7b-it.Q8_0.gguf,18-service69,3,format_fail,True,False,2,3,2,1,1,0,0.8,2,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.82,995,460,
gemma-7b-it.Q8_0.gguf,19-stecko-service,1,format_fail,True,False,2,2,1,0,0,0,0.7,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.12,1142,312,
gemma-7b-it.Q8_0.gguf,19-stecko-service,2,format_fail,True,False,2,2,1,0,0,0,0.6,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.36,1142,317,
gemma-7b-it.Q8_0.gguf,19-stecko-service,3,format_fail,True,False,2,3,2,1,1,0,0.8,1,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,15.13,1142,891,
gemma-7b-it.Q8_0.gguf,20-agym,1,format_fail,True,False,3,2,2,1,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,8.96,1075,483,
gemma-7b-it.Q8_0.gguf,20-agym,2,format_fail,True,False,3,2,1,0,0,0,0.6,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.27,1075,312,
gemma-7b-it.Q8_0.gguf,20-agym,3,format_fail,True,False,2,2,1,1,0,0,0.8,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.35,1075,435,
gemma-7b-it.Q8_0.gguf,21-my-fit,1,format_fail,True,False,2,3,2,1,1,0,0.8,1,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.65,1034,346,
gemma-7b-it.Q8_0.gguf,21-my-fit,2,format_fail,True,False,2,3,2,1,1,0,0.8,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.43,1034,441,
gemma-7b-it.Q8_0.gguf,21-my-fit,3,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,7.48,1034,443,
gemma-7b-it.Q8_0.gguf,22-lizavet,1,format_fail,True,False,2,1,2,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.94,859,315,
gemma-7b-it.Q8_0.gguf,22-lizavet,2,format_fail,True,False,2,2,1,0,0,0,0.8,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.67,859,397,
gemma-7b-it.Q8_0.gguf,22-lizavet,3,format_fail,True,False,2,1,2,0,0,0,0.6,1,0,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,5.46,859,326,
M8-saiga_gemma3_12b-Q8_0,09-geometriya-koles,2,format_fail,True,False,2,5,3,2,1,0,0.6,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,10.68,1052,935,
M8-saiga_gemma3_12b-Q8_0,09-geometriya-koles,3,ok,True,True,2,3,2,3,1,1,0.7,3,1,,1,40.69,1052,3540,
M8-saiga_gemma3_12b-Q8_0,10-krizanta,1,ok,True,True,2,2,1,1,1,1,0.5,3,1,,1,13.15,829,1105,
M8-saiga_gemma3_12b-Q8_0,10-krizanta,2,format_fail,True,False,2,2,1,2,1,0,0.5,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,38.94,829,3387,
M8-saiga_gemma3_12b-Q8_0,10-krizanta,3,ok,True,True,2,2,1,1,1,1,0.3,4,1,,1,20.98,829,1840,
M8-saiga_gemma3_12b-Q8_0,11-u-nails,1,ok,True,True,1,2,2,1,1,1,0.7,5,1,,1,21.63,849,1849,
M8-saiga_gemma3_12b-Q8_0,11-u-nails,2,format_fail,True,False,2,2,2,2,1,0,0.5,3,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,32.28,849,2818,
M8-saiga_gemma3_12b-Q8_0,11-u-nails,3,ok,True,True,2,2,2,2,1,1,0.5,5,1,,1,18.21,849,1599,
M8-saiga_gemma3_12b-Q8_0,12-parikmaherskaya-volokolamskaya,1,ok,True,True,1,2,1,1,1,1,0.5,5,1,,1,14.63,811,1238,
M8-saiga_gemma3_12b-Q8_0,12-parikmaherskaya-volokolamskaya,2,format_fail,True,False,2,2,1,1,1,0,0.5,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,23.01,811,2017,
M8-saiga_gemma3_12b-Q8_0,12-parikmaherskaya-volokolamskaya,3,ok,True,True,2,2,1,2,1,1,0.3,2,0,,1,16.98,811,1492,
M8-saiga_gemma3_12b-Q8_0,13-salon-radul,1,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,32.64,1089,2783,
M8-saiga_gemma3_12b-Q8_0,13-salon-radul,2,format_fail,True,False,2,3,2,1,2,0,0.8,7,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,27.85,1089,2447,
M8-saiga_gemma3_12b-Q8_0,13-salon-radul,3,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,11.19,1089,979,
M8-saiga_gemma3_12b-Q8_0,14-milliard-roz,1,ok,True,True,1,1,1,1,1,1,0.8,5,1,,1,15.16,1033,1269,
M8-saiga_gemma3_12b-Q8_0,14-milliard-roz,2,format_fail,True,False,1,2,1,1,1,0,0.6,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,32.93,1033,2864,
M8-saiga_gemma3_12b-Q8_0,14-milliard-roz,3,ok,True,True,2,2,1,1,1,1,0.5,5,0,,1,18.85,1033,1649,
M8-saiga_gemma3_12b-Q8_0,15-cvety-bez-povoda,1,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,16.68,1005,1403,
M8-saiga_gemma3_12b-Q8_0,15-cvety-bez-povoda,2,format_fail,True,False,1,2,1,1,1,0,0.6,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,17.08,1005,1498,
M8-saiga_gemma3_12b-Q8_0,15-cvety-bez-povoda,3,ok,True,True,1,2,1,1,1,1,0.3,1,1,,1,31.35,1005,2736,
M8-saiga_gemma3_12b-Q8_0,16-kameliya,1,ok,True,True,2,2,2,2,2,1,0.7,6,1,,1,14.37,1160,1188,
M8-saiga_gemma3_12b-Q8_0,16-kameliya,2,format_fail,True,False,3,3,2,3,2,0,0.7,6,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,25.91,1160,2260,
M8-saiga_gemma3_12b-Q8_0,16-kameliya,3,ok,True,True,3,3,2,3,1,1,0.7,7,1,,1,25.19,1160,2206,
M8-saiga_gemma3_12b-Q8_0,17-ip-electronics,1,ok,True,True,2,3,2,1,1,1,0.8,5,1,,1,11.27,1148,928,
M8-saiga_gemma3_12b-Q8_0,17-ip-electronics,2,format_fail,True,False,2,2,1,1,1,0,0.5,6,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,13.56,1148,1192,
M8-saiga_gemma3_12b-Q8_0,17-ip-electronics,3,format_fail,False,False,,,,,,0,0.8,,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,46.96,1148,4096,
M8-saiga_gemma3_12b-Q8_0,18-service69,1,ok,True,True,1,2,1,1,1,1,0.8,3,0,,1,31.08,995,2671,
M8-saiga_gemma3_12b-Q8_0,18-service69,2,format_fail,True,False,1,2,1,1,1,0,0.6,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,24.79,995,2158,
M8-saiga_gemma3_12b-Q8_0,18-service69,3,ok,True,True,2,2,1,1,1,1,0.3,3,0,,1,18.05,995,1581,
M8-saiga_gemma3_12b-Q8_0,19-stecko-service,1,ok,True,True,1,2,1,2,1,1,0.8,3,1,,1,27.15,1136,2301,
M8-saiga_gemma3_12b-Q8_0,19-stecko-service,2,format_fail,True,False,2,2,2,1,1,0,0.6,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,22.14,1136,1933,
M8-saiga_gemma3_12b-Q8_0,19-stecko-service,3,ok,True,True,1,2,1,1,1,1,0.6,3,1,,1,11.3,1136,987,
M8-saiga_gemma3_12b-Q8_0,20-agym,1,ok,True,True,3,3,2,2,2,1,0.6,5,1,,1,21.86,1078,1851,
M8-saiga_gemma3_12b-Q8_0,20-agym,2,format_fail,True,False,2,2,2,2,2,0,0.8,6,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,24.15,1078,2108,
M8-saiga_gemma3_12b-Q8_0,20-agym,3,format_fail,True,False,3,3,3,3,2,0,0.6,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0; evidence[4].quote: пуст или не строка",1,10.82,1078,948,
M8-saiga_gemma3_12b-Q8_0,21-my-fit,1,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,13.62,1058,1134,
M8-saiga_gemma3_12b-Q8_0,21-my-fit,2,format_fail,True,False,2,2,2,2,1,0,0.6,4,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,37.47,1058,3256,
M8-saiga_gemma3_12b-Q8_0,21-my-fit,3,ok,True,True,2,2,2,1,1,1,0.6,4,1,,1,12.05,1058,1050,
M8-saiga_gemma3_12b-Q8_0,22-lizavet,1,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,12.31,884,1030,
M8-saiga_gemma3_12b-Q8_0,22-lizavet,2,format_fail,True,False,1,2,1,1,1,0,0.3,5,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,37.37,884,3255,
M8-saiga_gemma3_12b-Q8_0,22-lizavet,3,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,20.16,884,1768,
M8-saiga_gemma3_12b-Q8_0,23-tver-kuhni,1,ok,True,True,1,1,1,1,1,1,0.5,3,1,,1,12.65,821,1065,
M8-saiga_gemma3_12b-Q8_0,23-tver-kuhni,2,format_fail,True,False,1,1,1,1,1,0,0.5,3,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,6.16,821,533,
M8-saiga_gemma3_12b-Q8_0,23-tver-kuhni,3,ok,True,True,1,1,1,1,1,1,0.3,4,1,,1,20.93,821,1848,
M8-saiga_gemma3_12b-Q8_0,24-rustekstil,1,ok,True,True,1,1,1,1,1,1,0.8,5,1,,1,31.67,827,2722,
M8-saiga_gemma3_12b-Q8_0,24-rustekstil,2,format_fail,True,False,1,2,1,1,1,0,0.4,3,1,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,16.31,827,1430,
M8-saiga_gemma3_12b-Q8_0,24-rustekstil,3,ok,True,True,2,2,1,1,1,1,0.3,2,1,,1,19.55,827,1720,"""

df_summary = pd.read_csv(io.StringIO(_CSV_DF_SUMMARY_BODY))
df_summary.shape
df_summary.head()


,model,dossier,run,status,parse_ok,ok,level,online_presence,process_automation,ecommerce_online_services,data_driven,ai_usage,confidence,n_evidence,quote_not_found,errors,attempts,total_s,prompt_tokens,completion_tokens,error
0,M10-GLM-4.7-Flash-Q6_K,01-ponchik,1,no_json,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,JSON не найден в ответе,1,80.49,939,4096,NaN
1,M10-GLM-4.7-Flash-Q6_K,01-ponchik,2,ok,True,True,3.0,3.0,3.0,5.0,3.0,1.0,0.9,4.0,1,NaN,1,54.67,939,2873,NaN
2,M10-GLM-4.7-Flash-Q6_K,01-ponchik,3,ok,True,True,3.0,3.0,3.0,5.0,3.0,1.0,0.9,4.0,1,NaN,1,54.71,939,2873,NaN
3,M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,1,ok,True,True,2.0,2.0,2.0,2.0,1.0,1.0,0.9,3.0,1,NaN,1,48.56,775,2555,NaN
4,M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,2,ok,True,True,2.0,4.0,1.0,3.0,1.0,1.0,0.4,3.0,1,NaN,1,65.84,775,3450,NaN


In [3]:
# summary_clean.csv — Только успешные прогоны + level_rationale
# CSV встроено напрямую в файл ноутбука

_CSV_DF_SUMMARY_CLEAN_BODY = r"""model,dossier,run,status,parse_ok,ok,level,online_presence,process_automation,ecommerce_online_services,data_driven,ai_usage,confidence,n_evidence,quote_not_found,errors,attempts,total_s,prompt_tokens,completion_tokens,error,level_rationale
M10-GLM-4.7-Flash-Q6_K,01-ponchik,2,ok,True,True,3,3,3,5,3,1,0.9,4,1,,1,54.67,939,2873,,
M10-GLM-4.7-Flash-Q6_K,01-ponchik,3,ok,True,True,3,3,3,5,3,1,0.9,4,1,,1,54.71,939,2873,,
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,1,ok,True,True,2,2,2,2,1,1,0.9,3,1,,1,48.56,775,2555,,
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,2,ok,True,True,2,4,1,3,1,1,0.4,3,1,,1,65.84,775,3450,,
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,3,ok,True,True,2,4,1,3,1,1,0.4,3,1,,1,65.86,775,3450,,
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,2,ok,True,True,3,3,3,4,3,1,0.9,4,1,,1,73.68,925,3827,,
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,3,ok,True,True,3,3,3,4,3,1,0.9,4,1,,1,73.7,925,3827,,
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,1,ok,True,True,2,4,1,2,1,1,0.8,4,0,,1,45.47,789,2398,,
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,2,ok,True,True,2,3,2,2,2,1,0.9,4,1,,1,53.58,789,2827,,
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,3,ok,True,True,2,3,2,2,2,1,0.9,4,1,,1,53.55,789,2827,,
M10-GLM-4.7-Flash-Q6_K,05-moncafe,1,ok,True,True,2,2,2,2,1,1,0.9,4,1,,1,37.55,830,1984,,
M10-GLM-4.7-Flash-Q6_K,05-moncafe,2,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,55.38,830,2916,,
M10-GLM-4.7-Flash-Q6_K,05-moncafe,3,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,55.39,830,2916,,
M10-GLM-4.7-Flash-Q6_K,06-breakfast-band,1,ok,True,True,2,2,2,2,2,1,0.8,4,0,,1,55.96,838,2926,,
M10-GLM-4.7-Flash-Q6_K,07-ru-service,1,ok,True,True,2,2,2,1,1,1,0.85,4,1,,1,60.91,753,3182,,
M10-GLM-4.7-Flash-Q6_K,07-ru-service,2,ok,True,True,2,4,2,2,2,1,0.8,3,0,,1,49.14,753,2602,,
M10-GLM-4.7-Flash-Q6_K,07-ru-service,3,ok,True,True,2,4,2,2,2,1,0.8,3,0,,1,49.22,753,2602,,
M10-GLM-4.7-Flash-Q6_K,08-avtobat,1,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,41.52,784,2190,,
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,2,ok,True,True,3,5,4,5,3,1,0.85,4,1,,1,68.22,1023,3628,,
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,3,ok,True,True,3,5,4,5,3,1,0.85,4,1,,1,68.17,1023,3628,,
M10-GLM-4.7-Flash-Q6_K,10-krizanta,1,ok,True,True,2,3,2,2,2,1,0.7,3,1,,1,46.25,775,2440,,
M10-GLM-4.7-Flash-Q6_K,11-u-nails,1,ok,True,True,2,3,2,2,2,1,0.8,3,1,,1,65.69,795,3424,,
M10-GLM-4.7-Flash-Q6_K,11-u-nails,2,ok,True,True,3,3,2,3,2,1,0.8,4,1,,1,68.3,795,3574,,
M10-GLM-4.7-Flash-Q6_K,11-u-nails,3,ok,True,True,3,3,2,3,2,1,0.8,4,1,,1,68.31,795,3574,,
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,1,ok,True,True,3,3,2,3,2,1,0.8,4,1,,1,67.65,757,3528,,
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,2,ok,True,True,3,4,3,3,2,1,0.6,3,0,,1,51.87,757,2744,,
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,3,ok,True,True,3,4,3,3,2,1,0.6,3,0,,1,51.83,757,2744,,
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,1,ok,True,True,1,2,1,1,1,1,0.9,3,0,,1,57.86,952,3013,,
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,2,ok,True,True,1,2,1,1,1,1,0.95,3,0,,1,62.52,952,3272,,
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,3,ok,True,True,1,2,1,1,1,1,0.95,3,0,,1,62.58,952,3272,,
M10-GLM-4.7-Flash-Q6_K,16-kameliya,1,ok,True,True,3,5,3,5,3,1,0.9,5,0,,1,74.98,1119,3850,,
M10-GLM-4.7-Flash-Q6_K,16-kameliya,2,ok,True,True,3,5,4,5,3,1,0.95,4,0,,1,45.07,1119,2375,,
M10-GLM-4.7-Flash-Q6_K,16-kameliya,3,ok,True,True,3,5,4,5,3,1,0.95,4,0,,1,45.01,1119,2375,,
M10-GLM-4.7-Flash-Q6_K,17-ip-electronics,1,ok,True,True,2,3,2,2,2,1,0.9,4,1,,1,53.67,1114,2778,,
M10-GLM-4.7-Flash-Q6_K,18-service69,2,ok,True,True,2,4,2,1,2,1,0.9,4,1,,1,47.11,958,2486,,
M10-GLM-4.7-Flash-Q6_K,18-service69,3,ok,True,True,2,4,2,1,2,1,0.9,4,1,,1,47.05,958,2486,,
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,1,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,62.06,1102,3201,,
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,2,ok,True,True,2,2,2,2,1,1,0.8,4,1,,1,51.63,1102,2711,,
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,3,ok,True,True,2,2,2,2,1,1,0.8,4,1,,1,51.6,1102,2711,,
M10-GLM-4.7-Flash-Q6_K,20-agym,1,ok,True,True,3,5,4,4,2,1,0.9,4,0,,1,43.88,1039,2287,,
M10-GLM-4.7-Flash-Q6_K,20-agym,2,ok,True,True,4,5,4,5,4,1,0.9,4,0,,1,60.32,1039,3178,,
M10-GLM-4.7-Flash-Q6_K,20-agym,3,ok,True,True,3,5,4,5,2,1,0.9,5,1,,1,45.57,1039,2458,,
M10-GLM-4.7-Flash-Q6_K,21-my-fit,2,ok,True,True,3,4,4,3,3,1,0.85,5,1,,1,46.25,1012,2492,,
M10-GLM-4.7-Flash-Q6_K,21-my-fit,3,ok,True,True,3,4,4,3,3,1,0.85,5,1,,1,46.17,1012,2492,,
M10-GLM-4.7-Flash-Q6_K,22-lizavet,1,ok,True,True,2,2,2,2,2,1,0.8,4,0,,1,70.38,834,3656,,
M10-GLM-4.7-Flash-Q6_K,23-tver-kuhni,1,ok,True,True,2,2,1,1,1,1,0.8,3,0,,1,67.98,769,3544,,
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,2,ok,True,True,2,2,2,1,2,1,0.8,4,0,,1,47.36,774,2512,,
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,3,ok,True,True,2,2,2,1,2,1,0.8,4,0,,1,47.27,774,2512,,
M11-gemma-4-31B-it-Q6_K,01-ponchik,1,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,94.11,975,1480,,
M12-T-pro-it-2.1-Q6_K,04-buna-buna,1,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,17.41,777,273,,
M12-T-pro-it-2.1-Q6_K,04-buna-buna,2,ok,True,True,2,2,2,1,1,1,0.85,5,1,,1,16.24,777,278,,
M12-T-pro-it-2.1-Q6_K,04-buna-buna,3,ok,True,True,2,2,2,1,1,1,0.85,5,1,,1,16.18,777,278,,
M12-T-pro-it-2.1-Q6_K,05-moncafe,1,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,19.3,816,303,,
M12-T-pro-it-2.1-Q6_K,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,17.74,816,303,,
M12-T-pro-it-2.1-Q6_K,05-moncafe,3,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,17.72,816,303,,
M12-T-pro-it-2.1-Q6_K,07-ru-service,1,ok,True,True,1,1,1,1,1,1,0.9,3,0,,1,18.54,738,295,,
M12-T-pro-it-2.1-Q6_K,08-avtobat,1,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,16.66,775,260,,
M12-T-pro-it-2.1-Q6_K,08-avtobat,2,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,21.48,775,366,,
M12-T-pro-it-2.1-Q6_K,08-avtobat,3,ok,True,True,2,2,2,1,1,1,0.8,5,1,,1,21.57,775,366,,
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,1,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,24.4,1020,368,,
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,2,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,22.12,1020,368,,
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,3,ok,True,True,3,3,3,3,2,1,0.9,5,1,,1,22.15,1020,368,,
M12-T-pro-it-2.1-Q6_K,10-krizanta,1,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,15.24,760,235,,
M12-T-pro-it-2.1-Q6_K,10-krizanta,2,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,13.8,760,235,,
M12-T-pro-it-2.1-Q6_K,10-krizanta,3,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,13.86,760,235,,
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,18.95,742,299,,
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,2,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,14.37,742,244,,
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,3,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,14.41,742,244,,
M12-T-pro-it-2.1-Q6_K,13-salon-radul,1,ok,True,True,2,2,2,1,1,1,0.9,6,0,,1,26.92,1008,409,,
M12-T-pro-it-2.1-Q6_K,13-salon-radul,2,ok,True,True,2,2,2,1,1,1,0.9,6,0,,1,24.08,1008,401,,
M12-T-pro-it-2.1-Q6_K,13-salon-radul,3,ok,True,True,2,2,2,1,1,1,0.9,6,0,,1,24.1,1008,401,,
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,20.98,971,313,,
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,2,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.26,971,322,,
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,3,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.29,971,322,,
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,1,ok,True,True,2,2,2,1,1,1,0.9,7,1,,1,28.42,1110,427,,
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,2,ok,True,True,2,2,2,1,1,1,0.9,7,1,,1,26.28,1110,435,,
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,3,ok,True,True,2,2,2,1,1,1,0.9,7,1,,1,26.19,1110,435,,
M12-T-pro-it-2.1-Q6_K,18-service69,1,ok,True,True,2,2,2,1,1,1,0.85,4,1,,1,20.87,947,313,,
M12-T-pro-it-2.1-Q6_K,18-service69,2,ok,True,True,2,2,2,1,1,1,0.9,5,1,,1,22.13,947,369,,
M12-T-pro-it-2.1-Q6_K,18-service69,3,ok,True,True,2,2,2,1,1,1,0.9,5,1,,1,22.12,947,369,,
M12-T-pro-it-2.1-Q6_K,19-stecko-service,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,23.32,1101,343,,
M12-T-pro-it-2.1-Q6_K,19-stecko-service,2,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,20.66,1101,343,,
M12-T-pro-it-2.1-Q6_K,19-stecko-service,3,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,20.67,1101,343,,
M12-T-pro-it-2.1-Q6_K,20-agym,1,ok,True,True,3,3,3,3,2,1,0.9,7,1,,1,26.28,1023,395,,
M12-T-pro-it-2.1-Q6_K,20-agym,2,ok,True,True,3,3,3,3,2,1,0.9,7,1,,1,23.81,1023,395,,
M12-T-pro-it-2.1-Q6_K,20-agym,3,ok,True,True,3,3,3,3,2,1,0.9,7,1,,1,23.76,1023,395,,
M12-T-pro-it-2.1-Q6_K,21-my-fit,1,ok,True,True,3,3,3,3,1,1,0.9,5,1,,1,21.45,964,320,,
M12-T-pro-it-2.1-Q6_K,21-my-fit,2,ok,True,True,3,3,3,3,1,1,0.9,5,1,,1,20.19,964,337,,
M12-T-pro-it-2.1-Q6_K,21-my-fit,3,ok,True,True,3,3,3,3,1,1,0.9,5,1,,1,20.22,964,337,,
M12-T-pro-it-2.1-Q6_K,22-lizavet,1,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,16.98,804,261,,
M12-T-pro-it-2.1-Q6_K,22-lizavet,2,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.25,804,324,,
M12-T-pro-it-2.1-Q6_K,22-lizavet,3,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,19.25,804,324,,
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,1,ok,True,True,2,2,2,1,1,1,0.7,6,1,,1,23.88,757,372,,
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,2,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,16.9,757,286,,
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,3,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,16.93,757,286,,
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,1,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,47.63,859,3136,,
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,47.52,859,3136,,
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,3,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,47.26,859,3136,,
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,1,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,40.86,861,2685,,
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,2,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,40.72,861,2685,,
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,3,ok,True,True,2,2,2,2,1,1,0.8,4,0,,1,40.5,861,2685,,
M7-gpt-oss-20b-Q8_0,01-ponchik,1,ok,True,True,3,2,2,3,2,1,0.7,5,1,,1,29.57,991,2479,,
M7-gpt-oss-20b-Q8_0,01-ponchik,2,ok,True,True,3,3,2,3,2,1,0.75,6,1,,1,19.99,991,1750,,
M7-gpt-oss-20b-Q8_0,01-ponchik,3,ok,True,True,3,2,2,3,2,1,0.9,5,1,,1,17.45,991,1545,,
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,1,ok,True,True,2,2,1,2,1,1,0.5,2,0,,1,15.34,829,1311,,
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,2,ok,True,True,1,2,1,2,1,1,0.6,5,1,,1,22.08,829,1946,,
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,3,ok,True,True,1,2,1,1,1,1,0.45,3,1,,1,17.72,829,1555,,
M7-gpt-oss-20b-Q8_0,03-surf-coffee,1,ok,True,True,1,2,2,1,1,1,0.4,4,1,,1,21.04,965,1787,,
M7-gpt-oss-20b-Q8_0,03-surf-coffee,2,ok,True,True,2,5,2,3,1,1,0.7,3,1,,1,22.38,965,1956,,
M7-gpt-oss-20b-Q8_0,04-buna-buna,1,ok,True,True,2,2,2,1,1,1,0.7,4,0,,1,27.79,838,2381,,
M7-gpt-oss-20b-Q8_0,04-buna-buna,2,ok,True,True,1,2,2,1,1,1,0.5,4,0,,1,21.22,838,1859,,
M7-gpt-oss-20b-Q8_0,04-buna-buna,3,ok,True,True,2,2,2,1,1,1,0.6,2,1,,1,11.63,838,1022,,
M7-gpt-oss-20b-Q8_0,05-moncafe,1,ok,True,True,1,1,1,1,1,1,0.8,5,1,,1,10.38,882,859,,
M7-gpt-oss-20b-Q8_0,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.6,5,0,,1,20.13,882,1770,,
M7-gpt-oss-20b-Q8_0,05-moncafe,3,ok,True,True,2,2,2,1,1,1,0.6,3,1,,1,22.05,882,1935,,
M7-gpt-oss-20b-Q8_0,06-breakfast-band,1,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,20.63,882,1758,,
M7-gpt-oss-20b-Q8_0,06-breakfast-band,2,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,10.47,882,918,,
M7-gpt-oss-20b-Q8_0,06-breakfast-band,3,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,23.87,882,2090,,
M7-gpt-oss-20b-Q8_0,07-ru-service,2,ok,True,True,1,1,1,1,1,1,0.9,5,1,,1,18.56,807,1628,,
M7-gpt-oss-20b-Q8_0,07-ru-service,3,ok,True,True,1,2,1,1,1,1,0.7,4,0,,1,17.4,807,1529,,
M7-gpt-oss-20b-Q8_0,08-avtobat,1,ok,True,True,2,2,1,1,1,1,0.4,4,1,,1,14.45,834,1224,,
M7-gpt-oss-20b-Q8_0,08-avtobat,2,ok,True,True,2,2,1,1,1,1,0.6,3,1,,1,8.44,834,737,,
M7-gpt-oss-20b-Q8_0,08-avtobat,3,ok,True,True,2,2,1,2,1,1,0.6,4,1,,1,8.97,834,796,,
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,1,ok,True,True,2,2,2,2,1,1,0.9,3,1,,1,15.62,1052,1307,,
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,2,ok,True,True,2,5,3,2,1,1,0.7,5,1,,1,10.68,1052,935,,
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,3,ok,True,True,2,3,2,3,1,1,0.75,3,1,,1,40.69,1052,3540,,
M7-gpt-oss-20b-Q8_0,10-krizanta,1,ok,True,True,2,2,1,1,1,1,0.6,3,1,,1,13.15,829,1105,,
M7-gpt-oss-20b-Q8_0,10-krizanta,2,ok,True,True,2,2,1,2,1,1,0.55,5,1,,1,38.94,829,3387,,
M7-gpt-oss-20b-Q8_0,10-krizanta,3,ok,True,True,2,2,1,1,1,1,0.4,4,1,,1,20.98,829,1840,,
M7-gpt-oss-20b-Q8_0,11-u-nails,1,ok,True,True,1,2,2,1,1,1,0.75,5,1,,1,21.63,849,1849,,
M7-gpt-oss-20b-Q8_0,11-u-nails,2,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,32.28,849,2818,,
M7-gpt-oss-20b-Q8_0,11-u-nails,3,ok,True,True,2,2,2,2,1,1,0.6,5,1,,1,18.21,849,1599,,
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,1,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,14.63,811,1238,,
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,2,ok,True,True,2,2,1,1,1,1,0.6,5,1,,1,23.01,811,2017,,
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,3,ok,True,True,2,2,1,2,1,1,0.4,2,0,,1,16.98,811,1492,,
M7-gpt-oss-20b-Q8_0,13-salon-radul,1,ok,True,True,2,2,2,1,1,1,0.7,5,1,,1,32.64,1089,2783,,
M7-gpt-oss-20b-Q8_0,13-salon-radul,2,ok,True,True,2,3,2,1,2,1,0.85,7,1,,1,27.85,1089,2447,,
M7-gpt-oss-20b-Q8_0,13-salon-radul,3,ok,True,True,2,2,2,1,1,1,0.73,5,1,,1,11.19,1089,979,,
M7-gpt-oss-20b-Q8_0,14-milliard-roz,1,ok,True,True,1,1,1,1,1,1,0.92,5,1,,1,15.16,1033,1269,,
M7-gpt-oss-20b-Q8_0,14-milliard-roz,2,ok,True,True,1,2,1,1,1,1,0.7,5,1,,1,32.93,1033,2864,,
M7-gpt-oss-20b-Q8_0,14-milliard-roz,3,ok,True,True,2,2,1,1,1,1,0.6,5,0,,1,18.85,1033,1649,,
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,1,ok,True,True,1,2,1,1,1,1,0.68,5,1,,1,16.68,1005,1403,,
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,2,ok,True,True,1,2,1,1,1,1,0.7,5,1,,1,17.08,1005,1498,,
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,3,ok,True,True,1,2,1,1,1,1,0.45,1,1,,1,31.35,1005,2736,,
M7-gpt-oss-20b-Q8_0,16-kameliya,1,ok,True,True,2,2,2,2,2,1,0.8,6,1,,1,14.37,1160,1188,,
M7-gpt-oss-20b-Q8_0,16-kameliya,2,ok,True,True,3,3,2,3,2,1,0.8,6,1,,1,25.91,1160,2260,,
M7-gpt-oss-20b-Q8_0,16-kameliya,3,ok,True,True,3,3,2,3,1,1,0.8,7,1,,1,25.19,1160,2206,,
M7-gpt-oss-20b-Q8_0,17-ip-electronics,1,ok,True,True,2,3,2,1,1,1,0.85,5,1,,1,11.27,1148,928,,
M7-gpt-oss-20b-Q8_0,17-ip-electronics,2,ok,True,True,2,2,1,1,1,1,0.55,6,1,,1,13.56,1148,1192,,
M7-gpt-oss-20b-Q8_0,18-service69,1,ok,True,True,1,2,1,1,1,1,0.85,3,0,,1,31.08,995,2671,,
M7-gpt-oss-20b-Q8_0,18-service69,2,ok,True,True,1,2,1,1,1,1,0.68,4,1,,1,24.79,995,2158,,
M7-gpt-oss-20b-Q8_0,18-service69,3,ok,True,True,2,2,1,1,1,1,0.4,3,0,,1,18.05,995,1581,,
M7-gpt-oss-20b-Q8_0,19-stecko-service,1,ok,True,True,1,2,1,2,1,1,0.85,3,1,,1,27.15,1136,2301,,
M7-gpt-oss-20b-Q8_0,19-stecko-service,2,ok,True,True,2,2,2,1,1,1,0.7,4,1,,1,22.14,1136,1933,,
M7-gpt-oss-20b-Q8_0,19-stecko-service,3,ok,True,True,1,2,1,1,1,1,0.7,3,1,,1,11.3,1136,987,,
M7-gpt-oss-20b-Q8_0,20-agym,1,ok,True,True,3,3,2,2,2,1,0.7,5,1,,1,21.86,1078,1851,,
M7-gpt-oss-20b-Q8_0,20-agym,2,ok,True,True,2,2,2,2,2,1,0.92,6,1,,1,24.15,1078,2108,,
M7-gpt-oss-20b-Q8_0,21-my-fit,1,ok,True,True,2,2,2,1,1,1,0.7,5,1,,1,13.62,1058,1134,,
M7-gpt-oss-20b-Q8_0,21-my-fit,2,ok,True,True,2,2,2,2,1,1,0.7,4,1,,1,37.47,1058,3256,,
M7-gpt-oss-20b-Q8_0,21-my-fit,3,ok,True,True,2,2,2,1,1,1,0.7,4,1,,1,12.05,1058,1050,,
M7-gpt-oss-20b-Q8_0,22-lizavet,1,ok,True,True,2,2,2,1,1,1,0.75,3,1,,1,12.31,884,1030,,
M7-gpt-oss-20b-Q8_0,22-lizavet,2,ok,True,True,1,2,1,1,1,1,0.4,5,1,,1,37.37,884,3255,,
M7-gpt-oss-20b-Q8_0,22-lizavet,3,ok,True,True,2,2,2,2,1,1,0.65,3,1,,1,20.16,884,1768,,
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,1,ok,True,True,1,1,1,1,1,1,0.6,3,1,,1,12.65,821,1065,,
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,2,ok,True,True,1,1,1,1,1,1,0.6,3,1,,1,6.16,821,533,,
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,3,ok,True,True,1,1,1,1,1,1,0.45,4,1,,1,20.93,821,1848,,
M7-gpt-oss-20b-Q8_0,24-rustekstil,1,ok,True,True,1,1,1,1,1,1,0.9,5,1,,1,31.67,827,2722,,
M7-gpt-oss-20b-Q8_0,24-rustekstil,2,ok,True,True,1,2,1,1,1,1,0.5,3,1,,1,16.31,827,1430,,
M7-gpt-oss-20b-Q8_0,24-rustekstil,3,ok,True,True,2,2,1,1,1,1,0.45,2,1,,1,19.55,827,1720,,
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,1,ok,True,True,2,2,2,2,1,1,0.6,5,1,,1,10.32,781,336,,
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,2,ok,True,True,2,3,1,2,1,1,0.75,7,1,,1,10.78,781,406,,
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,3,ok,True,True,3,3,2,3,2,1,0.8,5,1,,1,9.71,781,364,,
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,2,ok,True,True,2,4,2,2,1,1,0.8,4,1,,1,8.81,806,327,,
M8-saiga_gemma3_12b-Q8_0,05-moncafe,1,ok,True,True,3,3,3,3,2,1,1,6,1,,1,10.75,849,350,,
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,1,ok,True,True,2,2,2,1,1,1,0.8,6,1,,1,10.48,851,340,,
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,3,ok,True,True,2,4,1,2,1,1,0.7,5,1,,1,10.45,851,392,,
M8-saiga_gemma3_12b-Q8_0,07-ru-service,1,ok,True,True,1,2,1,1,1,1,0.6,3,1,,1,8.55,773,270,,
M8-saiga_gemma3_12b-Q8_0,07-ru-service,2,ok,True,True,2,3,2,1,1,1,0.8,5,1,,1,8.76,773,326,,
M8-saiga_gemma3_12b-Q8_0,07-ru-service,3,ok,True,True,1,2,1,1,1,1,0.6,6,1,,1,8.97,773,335,,
M8-saiga_gemma3_12b-Q8_0,08-avtobat,1,ok,True,True,2,3,2,1,1,1,0.7,4,1,,1,9.96,802,321,,
M8-saiga_gemma3_12b-Q8_0,09-geometriya-koles,1,ok,True,True,3,4,3,3,3,1,1,6,1,,1,12.97,1051,427,,
M9-gemma-4-12b-it-Q8_0,01-ponchik,1,ok,True,True,3,4,3,4,2,1,0.95,5,1,,1,52.71,975,1944,,
M9-gemma-4-12b-it-Q8_0,01-ponchik,2,ok,True,True,3,2,3,3,2,1,0.9,6,0,,1,85.01,975,3214,,
M9-gemma-4-12b-it-Q8_0,01-ponchik,3,ok,True,True,3,3,3,3,3,1,0.9,6,1,,1,88.09,975,3334,,
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,1,ok,True,True,4,4,3,4,4,1,0.9,5,0,,1,51.41,951,1881,,
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,2,ok,True,True,4,4,3,4,4,1,0.85,6,0,,1,51.99,951,1961,,
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,3,ok,True,True,3,3,3,3,3,1,0.9,5,0,,1,66.54,951,2516,,
M9-gemma-4-12b-it-Q8_0,04-buna-buna,1,ok,True,True,2,2,2,1,1,1,0.85,3,0,,1,45.94,813,1682,,
M9-gemma-4-12b-it-Q8_0,04-buna-buna,2,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,39.17,813,1475,,
M9-gemma-4-12b-it-Q8_0,04-buna-buna,3,ok,True,True,2,2,2,1,1,1,0.8,3,1,,1,42.63,813,1606,,
M9-gemma-4-12b-it-Q8_0,05-moncafe,1,ok,True,True,2,2,2,1,1,1,0.8,4,1,,1,40.86,856,1485,,
M9-gemma-4-12b-it-Q8_0,05-moncafe,2,ok,True,True,2,2,2,1,1,1,0.7,4,1,,1,65.14,856,2464,,
M9-gemma-4-12b-it-Q8_0,05-moncafe,3,ok,True,True,2,2,1,1,1,1,0.9,4,0,,1,43.46,856,1636,,
M9-gemma-4-12b-it-Q8_0,06-breakfast-band,1,ok,True,True,2,2,2,1,1,1,0.7,3,0,,1,60.03,858,2216,,
M8-saiga_gemma3_12b-Q8_0,09-geometriya-koles,3,ok,True,True,2,3,2,3,1,1,0.7,3,1,,1,40.69,1052,3540,,
M8-saiga_gemma3_12b-Q8_0,10-krizanta,1,ok,True,True,2,2,1,1,1,1,0.5,3,1,,1,13.15,829,1105,,
M8-saiga_gemma3_12b-Q8_0,10-krizanta,3,ok,True,True,2,2,1,1,1,1,0.3,4,1,,1,20.98,829,1840,,
M8-saiga_gemma3_12b-Q8_0,11-u-nails,1,ok,True,True,1,2,2,1,1,1,0.7,5,1,,1,21.63,849,1849,,
M8-saiga_gemma3_12b-Q8_0,11-u-nails,3,ok,True,True,2,2,2,2,1,1,0.5,5,1,,1,18.21,849,1599,,
M8-saiga_gemma3_12b-Q8_0,12-parikmaherskaya-volokolamskaya,1,ok,True,True,1,2,1,1,1,1,0.5,5,1,,1,14.63,811,1238,,
M8-saiga_gemma3_12b-Q8_0,12-parikmaherskaya-volokolamskaya,3,ok,True,True,2,2,1,2,1,1,0.3,2,0,,1,16.98,811,1492,,
M8-saiga_gemma3_12b-Q8_0,13-salon-radul,1,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,32.64,1089,2783,,
M8-saiga_gemma3_12b-Q8_0,13-salon-radul,3,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,11.19,1089,979,,
M8-saiga_gemma3_12b-Q8_0,14-milliard-roz,1,ok,True,True,1,1,1,1,1,1,0.8,5,1,,1,15.16,1033,1269,,
M8-saiga_gemma3_12b-Q8_0,14-milliard-roz,3,ok,True,True,2,2,1,1,1,1,0.5,5,0,,1,18.85,1033,1649,,
M8-saiga_gemma3_12b-Q8_0,15-cvety-bez-povoda,1,ok,True,True,1,2,1,1,1,1,0.6,5,1,,1,16.68,1005,1403,,
M8-saiga_gemma3_12b-Q8_0,15-cvety-bez-povoda,3,ok,True,True,1,2,1,1,1,1,0.3,1,1,,1,31.35,1005,2736,,
M8-saiga_gemma3_12b-Q8_0,16-kameliya,1,ok,True,True,2,2,2,2,2,1,0.7,6,1,,1,14.37,1160,1188,,
M8-saiga_gemma3_12b-Q8_0,16-kameliya,3,ok,True,True,3,3,2,3,1,1,0.7,7,1,,1,25.19,1160,2206,,
M8-saiga_gemma3_12b-Q8_0,17-ip-electronics,1,ok,True,True,2,3,2,1,1,1,0.8,5,1,,1,11.27,1148,928,,
M8-saiga_gemma3_12b-Q8_0,18-service69,1,ok,True,True,1,2,1,1,1,1,0.8,3,0,,1,31.08,995,2671,,
M8-saiga_gemma3_12b-Q8_0,18-service69,3,ok,True,True,2,2,1,1,1,1,0.3,3,0,,1,18.05,995,1581,,
M8-saiga_gemma3_12b-Q8_0,19-stecko-service,1,ok,True,True,1,2,1,2,1,1,0.8,3,1,,1,27.15,1136,2301,,
M8-saiga_gemma3_12b-Q8_0,19-stecko-service,3,ok,True,True,1,2,1,1,1,1,0.6,3,1,,1,11.3,1136,987,,
M8-saiga_gemma3_12b-Q8_0,20-agym,1,ok,True,True,3,3,2,2,2,1,0.6,5,1,,1,21.86,1078,1851,,
M8-saiga_gemma3_12b-Q8_0,21-my-fit,1,ok,True,True,2,2,2,1,1,1,0.6,5,1,,1,13.62,1058,1134,,
M8-saiga_gemma3_12b-Q8_0,21-my-fit,3,ok,True,True,2,2,2,1,1,1,0.6,4,1,,1,12.05,1058,1050,,
M8-saiga_gemma3_12b-Q8_0,22-lizavet,1,ok,True,True,2,2,2,1,1,1,0.7,3,1,,1,12.31,884,1030,,
M8-saiga_gemma3_12b-Q8_0,22-lizavet,3,ok,True,True,2,2,2,2,1,1,0.6,3,1,,1,20.16,884,1768,,
M8-saiga_gemma3_12b-Q8_0,23-tver-kuhni,1,ok,True,True,1,1,1,1,1,1,0.5,3,1,,1,12.65,821,1065,,
M8-saiga_gemma3_12b-Q8_0,23-tver-kuhni,3,ok,True,True,1,1,1,1,1,1,0.3,4,1,,1,20.93,821,1848,,
M8-saiga_gemma3_12b-Q8_0,24-rustekstil,1,ok,True,True,1,1,1,1,1,1,0.8,5,1,,1,31.67,827,2722,,
M8-saiga_gemma3_12b-Q8_0,24-rustekstil,3,ok,True,True,2,2,1,1,1,1,0.3,2,1,,1,19.55,827,1720,,"""

df_summary_clean = pd.read_csv(io.StringIO(_CSV_DF_SUMMARY_CLEAN_BODY))
df_summary_clean.shape
df_summary_clean.head()


,model,dossier,run,status,parse_ok,ok,level,online_presence,process_automation,ecommerce_online_services,data_driven,ai_usage,confidence,n_evidence,quote_not_found,errors,attempts,total_s,prompt_tokens,completion_tokens,error,level_rationale
0,M10-GLM-4.7-Flash-Q6_K,01-ponchik,2,ok,True,True,3,3,3,5,3,1,0.9,4,1,NaN,1,54.67,939,2873,NaN,NaN
1,M10-GLM-4.7-Flash-Q6_K,01-ponchik,3,ok,True,True,3,3,3,5,3,1,0.9,4,1,NaN,1,54.71,939,2873,NaN,NaN
2,M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,1,ok,True,True,2,2,2,2,1,1,0.9,3,1,NaN,1,48.56,775,2555,NaN,NaN
3,M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,2,ok,True,True,2,4,1,3,1,1,0.4,3,1,NaN,1,65.84,775,3450,NaN,NaN
4,M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,3,ok,True,True,2,4,1,3,1,1,0.4,3,1,NaN,1,65.86,775,3450,NaN,NaN


In [4]:
# summary_comprehensive.csv — Расширенная валидация (ошибки/предупреждения)
# CSV встроено напрямую в файл ноутбука

_CSV_DF_SUMMARY_COMPREHENSIVE_BODY = r"""model,dossier,run,status,level,online_presence,process_automation,ecommerce_online_services,data_driven,ai_usage,confidence,n_evidence,has_level_rationale,has_evidence,zero_ai_usage,zero_data_driven,errors,warnings_count,attempts,total_s,prompt_tokens,completion_tokens
M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,1,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",7,1,6.42,1992,535
M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,2,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",7,1,4.18,1992,478
M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,3,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",7,1,4.18,1992,478
M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,1,format_fail,2,3,2,3,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.55,1819,372
M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,2,format_fail,2,2,1,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,2.54,1819,290
M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,3,format_fail,2,2,1,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,2.54,1819,290
M6-GigaChat3.1-10B-A1.8B-q8_0,03-surf-coffee,1,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",5,1,3.3,1973,331
M6-GigaChat3.1-10B-A1.8B-q8_0,03-surf-coffee,2,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,2.79,1973,319
M6-GigaChat3.1-10B-A1.8B-q8_0,03-surf-coffee,3,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,2.79,1973,319
M6-GigaChat3.1-10B-A1.8B-q8_0,04-buna-buna,1,format_fail,2,2,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.08,1848,312
M6-GigaChat3.1-10B-A1.8B-q8_0,04-buna-buna,2,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,3.08,1848,352
M6-GigaChat3.1-10B-A1.8B-q8_0,04-buna-buna,3,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,3.08,1848,352
M6-GigaChat3.1-10B-A1.8B-q8_0,05-moncafe,1,format_fail,2,2,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,3.45,1882,353
M6-GigaChat3.1-10B-A1.8B-q8_0,05-moncafe,2,format_fail,2,2,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",2,1,3.24,1882,371
M6-GigaChat3.1-10B-A1.8B-q8_0,05-moncafe,3,format_fail,2,2,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",2,1,3.24,1882,371
M6-GigaChat3.1-10B-A1.8B-q8_0,06-breakfast-band,1,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.29,1900,334
M6-GigaChat3.1-10B-A1.8B-q8_0,06-breakfast-band,2,format_fail,2,3,2,2,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.03,1900,347
M6-GigaChat3.1-10B-A1.8B-q8_0,06-breakfast-band,3,format_fail,2,3,2,2,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.03,1900,347
M6-GigaChat3.1-10B-A1.8B-q8_0,07-ru-service,1,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,4.03,1814,427
M6-GigaChat3.1-10B-A1.8B-q8_0,07-ru-service,2,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,2.91,1814,333
M6-GigaChat3.1-10B-A1.8B-q8_0,07-ru-service,3,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,2.91,1814,333
M6-GigaChat3.1-10B-A1.8B-q8_0,08-avtobat,1,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.39,1844,349
M6-GigaChat3.1-10B-A1.8B-q8_0,08-avtobat,2,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.12,1844,357
M6-GigaChat3.1-10B-A1.8B-q8_0,08-avtobat,3,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.12,1844,357
M6-GigaChat3.1-10B-A1.8B-q8_0,09-geometriya-koles,1,format_fail,3,4,4,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.75,2073,366
M6-GigaChat3.1-10B-A1.8B-q8_0,09-geometriya-koles,2,format_fail,3,4,4,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,3.69,2073,422
M6-GigaChat3.1-10B-A1.8B-q8_0,09-geometriya-koles,3,format_fail,3,4,4,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,3.69,2073,422
M6-GigaChat3.1-10B-A1.8B-q8_0,10-krizanta,1,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,2.8,1828,281
M6-GigaChat3.1-10B-A1.8B-q8_0,10-krizanta,2,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,1,2.25,1828,257
M6-GigaChat3.1-10B-A1.8B-q8_0,10-krizanta,3,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",1,1,2.25,1828,257
M6-GigaChat3.1-10B-A1.8B-q8_0,11-u-nails,1,format_fail,2,2,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,3.68,1854,382
M6-GigaChat3.1-10B-A1.8B-q8_0,11-u-nails,2,format_fail,2,2,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,2.63,1854,301
M6-GigaChat3.1-10B-A1.8B-q8_0,11-u-nails,3,format_fail,2,2,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,2.63,1854,301
M6-GigaChat3.1-10B-A1.8B-q8_0,12-parikmaherskaya-volokolamskaya,1,format_fail,2,2,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.53,1817,369
M6-GigaChat3.1-10B-A1.8B-q8_0,12-parikmaherskaya-volokolamskaya,2,format_fail,2,2,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,2.86,1817,327
M6-GigaChat3.1-10B-A1.8B-q8_0,12-parikmaherskaya-volokolamskaya,3,format_fail,2,2,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,2.86,1817,327
M6-GigaChat3.1-10B-A1.8B-q8_0,13-salon-radul,1,format_fail,2,4,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,4.31,2082,432
M6-GigaChat3.1-10B-A1.8B-q8_0,13-salon-radul,2,format_fail,2,4,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.63,2082,415
M6-GigaChat3.1-10B-A1.8B-q8_0,13-salon-radul,3,format_fail,2,4,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.62,2082,415
M6-GigaChat3.1-10B-A1.8B-q8_0,14-milliard-roz,1,format_fail,2,2,2,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.98,2041,395
M6-GigaChat3.1-10B-A1.8B-q8_0,14-milliard-roz,2,format_fail,2,2,2,1,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.69,2041,422
M6-GigaChat3.1-10B-A1.8B-q8_0,14-milliard-roz,3,format_fail,2,2,2,1,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.69,2041,422
M6-GigaChat3.1-10B-A1.8B-q8_0,15-cvety-bez-povoda,1,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.84,1988,390
M6-GigaChat3.1-10B-A1.8B-q8_0,15-cvety-bez-povoda,2,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.05,1988,348
M6-GigaChat3.1-10B-A1.8B-q8_0,15-cvety-bez-povoda,3,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.05,1988,348
M6-GigaChat3.1-10B-A1.8B-q8_0,16-kameliya,1,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",6,1,5.17,2172,518
M6-GigaChat3.1-10B-A1.8B-q8_0,16-kameliya,2,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",5,1,5.87,2172,669
M6-GigaChat3.1-10B-A1.8B-q8_0,16-kameliya,3,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",5,1,5.86,2172,669
M6-GigaChat3.1-10B-A1.8B-q8_0,17-ip-electronics,1,format_fail,3,4,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0; level_rationale: пуст или не строка",5,1,4.22,2168,409
M6-GigaChat3.1-10B-A1.8B-q8_0,17-ip-electronics,2,format_fail,3,4,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",8,1,4.1,2168,467
M6-GigaChat3.1-10B-A1.8B-q8_0,17-ip-electronics,3,format_fail,3,4,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",8,1,4.1,2168,467
M6-GigaChat3.1-10B-A1.8B-q8_0,18-service69,1,format_fail,2,3,2,1,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.59,2020,357
M6-GigaChat3.1-10B-A1.8B-q8_0,18-service69,2,format_fail,2,3,2,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,2.65,2020,301
M6-GigaChat3.1-10B-A1.8B-q8_0,18-service69,3,format_fail,2,3,2,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,2.66,2020,301
M6-GigaChat3.1-10B-A1.8B-q8_0,19-stecko-service,1,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.99,2152,388
M6-GigaChat3.1-10B-A1.8B-q8_0,19-stecko-service,2,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.78,2152,431
M6-GigaChat3.1-10B-A1.8B-q8_0,19-stecko-service,3,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.78,2152,431
M6-GigaChat3.1-10B-A1.8B-q8_0,20-agym,1,format_fail,3,4,3,4,2,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.73,2088,364
M6-GigaChat3.1-10B-A1.8B-q8_0,20-agym,2,format_fail,3,4,4,4,2,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.91,2088,447
M6-GigaChat3.1-10B-A1.8B-q8_0,20-agym,3,format_fail,3,4,4,4,2,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.9,2088,447
M6-GigaChat3.1-10B-A1.8B-q8_0,21-my-fit,1,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,3.73,2046,367
M6-GigaChat3.1-10B-A1.8B-q8_0,21-my-fit,2,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,3.61,2046,412
M6-GigaChat3.1-10B-A1.8B-q8_0,21-my-fit,3,format_fail,3,4,3,4,1,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,3.61,2046,412
M6-GigaChat3.1-10B-A1.8B-q8_0,22-lizavet,1,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,3.28,1880,334
M6-GigaChat3.1-10B-A1.8B-q8_0,22-lizavet,2,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.23,1880,370
M6-GigaChat3.1-10B-A1.8B-q8_0,22-lizavet,3,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,3.23,1880,370
M6-GigaChat3.1-10B-A1.8B-q8_0,23-tver-kuhni,1,format_fail,2,3,2,2,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,3.28,1831,337
M6-GigaChat3.1-10B-A1.8B-q8_0,23-tver-kuhni,2,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.11,1831,356
M6-GigaChat3.1-10B-A1.8B-q8_0,23-tver-kuhni,3,format_fail,2,3,2,2,0,0,0.9,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,3.11,1831,356
M6-GigaChat3.1-10B-A1.8B-q8_0,24-rustekstil,1,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,3.2,1837,327
M6-GigaChat3.1-10B-A1.8B-q8_0,24-rustekstil,2,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,2.91,1837,333
M6-GigaChat3.1-10B-A1.8B-q8_0,24-rustekstil,3,format_fail,2,2,1,1,0,0,1,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,2.91,1837,333
M7-gpt-oss-20b-Q8_0,01-ponchik,1,ok,3,2,2,3,2,1,0.7,0,0,0,0,0,,2,1,29.57,991,2479
M7-gpt-oss-20b-Q8_0,01-ponchik,2,ok,3,3,2,3,2,1,0.75,0,0,0,0,0,,4,1,19.99,991,1750
M7-gpt-oss-20b-Q8_0,01-ponchik,3,ok,3,2,2,3,2,1,0.9,0,0,0,0,0,,3,1,17.45,991,1545
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,1,ok,2,2,1,2,1,1,0.5,0,0,0,0,0,,0,1,15.34,829,1311
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,2,ok,1,2,1,2,1,1,0.6,0,0,0,0,0,,1,1,22.08,829,1946
M7-gpt-oss-20b-Q8_0,02-tabella-bakery,3,ok,1,2,1,1,1,1,0.45,0,0,0,0,0,,1,1,17.72,829,1555
M7-gpt-oss-20b-Q8_0,03-surf-coffee,1,ok,1,2,2,1,1,1,0.4,0,0,0,0,0,,2,1,21.04,965,1787
M7-gpt-oss-20b-Q8_0,03-surf-coffee,2,ok,2,5,2,3,1,1,0.7,0,0,0,0,0,,3,1,22.38,965,1956
M7-gpt-oss-20b-Q8_0,04-buna-buna,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,0,1,27.79,838,2381
M7-gpt-oss-20b-Q8_0,04-buna-buna,2,ok,1,2,2,1,1,1,0.5,0,0,0,0,0,,0,1,21.22,838,1859
M7-gpt-oss-20b-Q8_0,04-buna-buna,3,ok,2,2,2,1,1,1,0.6,0,0,0,0,0,,2,1,11.63,838,1022
M7-gpt-oss-20b-Q8_0,05-moncafe,1,ok,1,1,1,1,1,1,0.8,0,0,0,0,0,,5,1,10.38,882,859
M7-gpt-oss-20b-Q8_0,05-moncafe,2,ok,2,2,2,1,1,1,0.6,0,0,0,0,0,,0,1,20.13,882,1770
M7-gpt-oss-20b-Q8_0,05-moncafe,3,ok,2,2,2,1,1,1,0.6,0,0,0,0,0,,1,1,22.05,882,1935
M7-gpt-oss-20b-Q8_0,06-breakfast-band,1,ok,1,2,1,1,1,1,0.6,0,0,0,0,0,,3,1,20.63,882,1758
M7-gpt-oss-20b-Q8_0,06-breakfast-band,2,ok,2,2,2,1,1,1,0.6,0,0,0,0,0,,3,1,10.47,882,918
M7-gpt-oss-20b-Q8_0,06-breakfast-band,3,ok,1,2,1,1,1,1,0.6,0,0,0,0,0,,4,1,23.87,882,2090
M7-gpt-oss-20b-Q8_0,07-ru-service,2,ok,1,1,1,1,1,1,0.9,0,0,0,0,0,,5,1,18.56,807,1628
M7-gpt-oss-20b-Q8_0,07-ru-service,3,ok,1,2,1,1,1,1,0.7,0,0,0,0,0,,0,1,17.4,807,1529
M7-gpt-oss-20b-Q8_0,08-avtobat,1,ok,2,2,1,1,1,1,0.4,0,0,0,0,0,,1,1,14.45,834,1224
M7-gpt-oss-20b-Q8_0,08-avtobat,2,ok,2,2,1,1,1,1,0.6,0,0,0,0,0,,3,1,8.44,834,737
M7-gpt-oss-20b-Q8_0,08-avtobat,3,ok,2,2,1,2,1,1,0.6,0,0,0,0,0,,2,1,8.97,834,796
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,1,ok,2,2,2,2,1,1,0.9,0,0,0,0,0,,3,1,15.62,1052,1307
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,2,ok,2,5,3,2,1,1,0.7,0,0,0,0,0,,5,1,10.68,1052,935
M7-gpt-oss-20b-Q8_0,09-geometriya-koles,3,ok,2,3,2,3,1,1,0.75,0,0,0,0,0,,1,1,40.69,1052,3540
M7-gpt-oss-20b-Q8_0,10-krizanta,1,ok,2,2,1,1,1,1,0.6,0,0,0,0,0,,1,1,13.15,829,1105
M7-gpt-oss-20b-Q8_0,10-krizanta,2,ok,2,2,1,2,1,1,0.55,0,0,0,0,0,,2,1,38.94,829,3387
M7-gpt-oss-20b-Q8_0,10-krizanta,3,ok,2,2,1,1,1,1,0.4,0,0,0,0,0,,1,1,20.98,829,1840
M7-gpt-oss-20b-Q8_0,11-u-nails,1,ok,1,2,2,1,1,1,0.75,0,0,0,0,0,,5,1,21.63,849,1849
M7-gpt-oss-20b-Q8_0,11-u-nails,2,ok,2,2,2,2,1,1,0.6,0,0,0,0,0,,3,1,32.28,849,2818
M7-gpt-oss-20b-Q8_0,11-u-nails,3,ok,2,2,2,2,1,1,0.6,0,0,0,0,0,,5,1,18.21,849,1599
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,1,ok,1,2,1,1,1,1,0.6,0,0,0,0,0,,3,1,14.63,811,1238
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,2,ok,2,2,1,1,1,1,0.6,0,0,0,0,0,,3,1,23.01,811,2017
M7-gpt-oss-20b-Q8_0,12-parikmaherskaya-volokolamskaya,3,ok,2,2,1,2,1,1,0.4,0,0,0,0,0,,0,1,16.98,811,1492
M7-gpt-oss-20b-Q8_0,13-salon-radul,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,3,1,32.64,1089,2783
M7-gpt-oss-20b-Q8_0,13-salon-radul,2,ok,2,3,2,1,2,1,0.85,0,0,0,0,0,,3,1,27.85,1089,2447
M7-gpt-oss-20b-Q8_0,13-salon-radul,3,ok,2,2,2,1,1,1,0.73,0,0,0,0,0,,3,1,11.19,1089,979
M7-gpt-oss-20b-Q8_0,14-milliard-roz,1,ok,1,1,1,1,1,1,0.92,0,0,0,0,0,,5,1,15.16,1033,1269
M7-gpt-oss-20b-Q8_0,14-milliard-roz,2,ok,1,2,1,1,1,1,0.7,0,0,0,0,0,,4,1,32.93,1033,2864
M7-gpt-oss-20b-Q8_0,14-milliard-roz,3,ok,2,2,1,1,1,1,0.6,0,0,0,0,0,,0,1,18.85,1033,1649
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,1,ok,1,2,1,1,1,1,0.68,0,0,0,0,0,,3,1,16.68,1005,1403
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,2,ok,1,2,1,1,1,1,0.7,0,0,0,0,0,,5,1,17.08,1005,1498
M7-gpt-oss-20b-Q8_0,15-cvety-bez-povoda,3,ok,1,2,1,1,1,1,0.45,0,0,0,0,0,,1,1,31.35,1005,2736
M7-gpt-oss-20b-Q8_0,16-kameliya,1,ok,2,2,2,2,2,1,0.8,0,0,0,0,0,,4,1,14.37,1160,1188
M7-gpt-oss-20b-Q8_0,16-kameliya,2,ok,3,3,2,3,2,1,0.8,0,0,0,0,0,,1,1,25.91,1160,2260
M7-gpt-oss-20b-Q8_0,16-kameliya,3,ok,3,3,2,3,1,1,0.8,0,0,0,0,0,,1,1,25.19,1160,2206
M7-gpt-oss-20b-Q8_0,17-ip-electronics,1,ok,2,3,2,1,1,1,0.85,0,0,0,0,0,,4,1,11.27,1148,928
M7-gpt-oss-20b-Q8_0,17-ip-electronics,2,ok,2,2,1,1,1,1,0.55,0,0,0,0,0,,6,1,13.56,1148,1192
M7-gpt-oss-20b-Q8_0,18-service69,1,ok,1,2,1,1,1,1,0.85,0,0,0,0,0,,0,1,31.08,995,2671
M7-gpt-oss-20b-Q8_0,18-service69,2,ok,1,2,1,1,1,1,0.68,0,0,0,0,0,,4,1,24.79,995,2158
M7-gpt-oss-20b-Q8_0,18-service69,3,ok,2,2,1,1,1,1,0.4,0,0,0,0,0,,0,1,18.05,995,1581
M7-gpt-oss-20b-Q8_0,19-stecko-service,1,ok,1,2,1,2,1,1,0.85,0,0,0,0,0,,1,1,27.15,1136,2301
M7-gpt-oss-20b-Q8_0,19-stecko-service,2,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,4,1,22.14,1136,1933
M7-gpt-oss-20b-Q8_0,19-stecko-service,3,ok,1,2,1,1,1,1,0.7,0,0,0,0,0,,1,1,11.3,1136,987
M7-gpt-oss-20b-Q8_0,20-agym,1,ok,3,3,2,2,2,1,0.7,0,0,0,0,0,,1,1,21.86,1078,1851
M7-gpt-oss-20b-Q8_0,20-agym,2,ok,2,2,2,2,2,1,0.92,0,0,0,0,0,,3,1,24.15,1078,2108
M7-gpt-oss-20b-Q8_0,20-agym,3,format_fail,3,3,3,3,2,0,0.7,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0; evidence[4].quote: пуст или не строка",2,1,10.82,1078,948
M7-gpt-oss-20b-Q8_0,21-my-fit,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,5,1,13.62,1058,1134
M7-gpt-oss-20b-Q8_0,21-my-fit,2,ok,2,2,2,2,1,1,0.7,0,0,0,0,0,,3,1,37.47,1058,3256
M7-gpt-oss-20b-Q8_0,21-my-fit,3,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,1,1,12.05,1058,1050
M7-gpt-oss-20b-Q8_0,22-lizavet,1,ok,2,2,2,1,1,1,0.75,0,0,0,0,0,,3,1,12.31,884,1030
M7-gpt-oss-20b-Q8_0,22-lizavet,2,ok,1,2,1,1,1,1,0.4,0,0,0,0,0,,1,1,37.37,884,3255
M7-gpt-oss-20b-Q8_0,22-lizavet,3,ok,2,2,2,2,1,1,0.65,0,0,0,0,0,,2,1,20.16,884,1768
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,1,ok,1,1,1,1,1,1,0.6,0,0,0,0,0,,1,1,12.65,821,1065
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,2,ok,1,1,1,1,1,1,0.6,0,0,0,0,0,,1,1,6.16,821,533
M7-gpt-oss-20b-Q8_0,23-tver-kuhni,3,ok,1,1,1,1,1,1,0.45,0,0,0,0,0,,1,1,20.93,821,1848
M7-gpt-oss-20b-Q8_0,24-rustekstil,1,ok,1,1,1,1,1,1,0.9,0,0,0,0,0,,3,1,31.67,827,2722
M7-gpt-oss-20b-Q8_0,24-rustekstil,2,ok,1,2,1,1,1,1,0.5,0,0,0,0,0,,1,1,16.31,827,1430
M7-gpt-oss-20b-Q8_0,24-rustekstil,3,ok,2,2,1,1,1,1,0.45,0,0,0,0,0,,1,1,19.55,827,1720
M8-saiga_gemma3_12b-Q8_0,01-ponchik,1,format_fail,3,3,3,4,3,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,11.97,968,405
M8-saiga_gemma3_12b-Q8_0,01-ponchik,2,format_fail,3,3,3,4,2,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,10.43,968,388
M8-saiga_gemma3_12b-Q8_0,01-ponchik,3,format_fail,3,4,2,4,3,0,0.9,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",2,1,8.97,968,331
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,1,ok,2,2,2,2,1,1,0.6,0,0,0,0,0,,2,1,10.32,781,336
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,2,ok,2,3,1,2,1,1,0.75,0,0,0,0,0,,3,1,10.78,781,406
M8-saiga_gemma3_12b-Q8_0,02-tabella-bakery,3,ok,3,3,2,3,2,1,0.8,0,0,0,0,0,,2,1,9.71,781,364
M8-saiga_gemma3_12b-Q8_0,03-surf-coffee,1,format_fail,3,4,3,4,3,0,0.9,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,11.49,944,373
M8-saiga_gemma3_12b-Q8_0,03-surf-coffee,2,format_fail,4,4,3,3,4,0,1,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,9.99,944,372
M8-saiga_gemma3_12b-Q8_0,03-surf-coffee,3,format_fail,3,4,2,3,1,0,0.9,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",4,1,10.24,944,382
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,1,format_fail,3,4,2,3,2,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,11.46,806,381
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,2,ok,2,4,2,2,1,1,0.8,0,0,0,0,0,,2,1,8.81,806,327
M8-saiga_gemma3_12b-Q8_0,04-buna-buna,3,format_fail,2,4,2,2,1,0,0.9,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,8.53,806,316
M8-saiga_gemma3_12b-Q8_0,05-moncafe,1,ok,3,3,3,3,2,1,1,0,0,0,0,0,,3,1,10.75,849,350
M8-saiga_gemma3_12b-Q8_0,05-moncafe,2,format_fail,2,2,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,7.57,849,278
M8-saiga_gemma3_12b-Q8_0,05-moncafe,3,format_fail,2,4,1,1,0,0,0.6,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",7,1,9.83,849,367
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,6,1,10.48,851,340
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,2,format_fail,2,3,1,2,1,0,0.7,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,8.03,851,296
M8-saiga_gemma3_12b-Q8_0,06-breakfast-band,3,ok,2,4,1,2,1,1,0.7,0,0,0,0,0,,3,1,10.45,851,392
M8-saiga_gemma3_12b-Q8_0,07-ru-service,1,ok,1,2,1,1,1,1,0.6,0,0,0,0,0,,1,1,8.55,773,270
M8-saiga_gemma3_12b-Q8_0,07-ru-service,2,ok,2,3,2,1,1,1,0.8,0,0,0,0,0,,4,1,8.76,773,326
M8-saiga_gemma3_12b-Q8_0,07-ru-service,3,ok,1,2,1,1,1,1,0.6,0,0,0,0,0,,2,1,8.97,773,335
M8-saiga_gemma3_12b-Q8_0,08-avtobat,1,ok,2,3,2,1,1,1,0.7,0,0,0,0,0,,1,1,9.96,802,321
M8-saiga_gemma3_12b-Q8_0,08-avtobat,2,format_fail,3,3,2,3,0,0,0.7,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",5,1,11.56,802,436
M8-saiga_gemma3_12b-Q8_0,08-avtobat,3,format_fail,2,3,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",3,1,9.14,802,341
M8-saiga_gemma3_12b-Q8_0,09-geometriya-koles,1,ok,3,4,3,3,3,1,1,0,0,0,0,0,,5,1,12.97,1051,427
M9-gemma-4-12b-it-Q8_0,01-ponchik,1,ok,3,4,3,4,2,1,0.95,0,0,0,0,0,,1,1,52.71,975,1944
M9-gemma-4-12b-it-Q8_0,01-ponchik,2,ok,3,2,3,3,2,1,0.9,0,0,0,0,0,,0,1,85.01,975,3214
M9-gemma-4-12b-it-Q8_0,01-ponchik,3,ok,3,3,3,3,3,1,0.9,0,0,0,0,0,,2,1,88.09,975,3334
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,1,ok,4,4,3,4,4,1,0.9,0,0,0,0,0,,0,1,51.41,951,1881
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,2,ok,4,4,3,4,4,1,0.85,0,0,0,0,0,,0,1,51.99,951,1961
M9-gemma-4-12b-it-Q8_0,03-surf-coffee,3,ok,3,3,3,3,3,1,0.9,0,0,0,0,0,,0,1,66.54,951,2516
M9-gemma-4-12b-it-Q8_0,04-buna-buna,1,ok,2,2,2,1,1,1,0.85,0,0,0,0,0,,0,1,45.94,813,1682
M9-gemma-4-12b-it-Q8_0,04-buna-buna,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,39.17,813,1475
M9-gemma-4-12b-it-Q8_0,04-buna-buna,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,42.63,813,1606
M9-gemma-4-12b-it-Q8_0,05-moncafe,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,40.86,856,1485
M9-gemma-4-12b-it-Q8_0,05-moncafe,2,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,1,1,65.14,856,2464
M9-gemma-4-12b-it-Q8_0,05-moncafe,3,ok,2,2,1,1,1,1,0.9,0,0,0,0,0,,0,1,43.46,856,1636
M9-gemma-4-12b-it-Q8_0,06-breakfast-band,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,0,1,60.03,858,2216
M10-GLM-4.7-Flash-Q6_K,01-ponchik,2,ok,3,3,3,5,3,1,0.9,0,0,0,0,0,,1,1,54.67,939,2873
M10-GLM-4.7-Flash-Q6_K,01-ponchik,3,ok,3,3,3,5,3,1,0.9,0,0,0,0,0,,1,1,54.71,939,2873
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,1,ok,2,2,2,2,1,1,0.9,0,0,0,0,0,,1,1,48.56,775,2555
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,2,ok,2,4,1,3,1,1,0.4,0,0,0,0,0,,2,1,65.84,775,3450
M10-GLM-4.7-Flash-Q6_K,02-tabella-bakery,3,ok,2,4,1,3,1,1,0.4,0,0,0,0,0,,2,1,65.86,775,3450
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,2,ok,3,3,3,4,3,1,0.9,0,0,0,0,0,,1,1,73.68,925,3827
M10-GLM-4.7-Flash-Q6_K,03-surf-coffee,3,ok,3,3,3,4,3,1,0.9,0,0,0,0,0,,1,1,73.7,925,3827
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,1,ok,2,4,1,2,1,1,0.8,0,0,0,0,0,,0,1,45.47,789,2398
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,2,ok,2,3,2,2,2,1,0.9,0,0,0,0,0,,2,1,53.58,789,2827
M10-GLM-4.7-Flash-Q6_K,04-buna-buna,3,ok,2,3,2,2,2,1,0.9,0,0,0,0,0,,2,1,53.55,789,2827
M10-GLM-4.7-Flash-Q6_K,05-moncafe,1,ok,2,2,2,2,1,1,0.9,0,0,0,0,0,,1,1,37.55,830,1984
M10-GLM-4.7-Flash-Q6_K,05-moncafe,2,ok,2,2,2,2,1,1,0.6,0,0,0,0,0,,1,1,55.38,830,2916
M10-GLM-4.7-Flash-Q6_K,05-moncafe,3,ok,2,2,2,2,1,1,0.6,0,0,0,0,0,,1,1,55.39,830,2916
M10-GLM-4.7-Flash-Q6_K,06-breakfast-band,1,ok,2,2,2,2,2,1,0.8,0,0,0,0,0,,0,1,55.96,838,2926
M10-GLM-4.7-Flash-Q6_K,07-ru-service,1,ok,2,2,2,1,1,1,0.85,0,0,0,0,0,,1,1,60.91,753,3182
M10-GLM-4.7-Flash-Q6_K,07-ru-service,2,ok,2,4,2,2,2,1,0.8,0,0,0,0,0,,0,1,49.14,753,2602
M10-GLM-4.7-Flash-Q6_K,07-ru-service,3,ok,2,4,2,2,2,1,0.8,0,0,0,0,0,,0,1,49.22,753,2602
M10-GLM-4.7-Flash-Q6_K,08-avtobat,1,ok,2,2,2,2,1,1,0.6,0,0,0,0,0,,3,1,41.52,784,2190
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,2,ok,3,5,4,5,3,1,0.85,0,0,0,0,0,,3,1,68.22,1023,3628
M10-GLM-4.7-Flash-Q6_K,09-geometriya-koles,3,ok,3,5,4,5,3,1,0.85,0,0,0,0,0,,3,1,68.17,1023,3628
M10-GLM-4.7-Flash-Q6_K,10-krizanta,1,ok,2,3,2,2,2,1,0.7,0,0,0,0,0,,1,1,46.25,775,2440
M10-GLM-4.7-Flash-Q6_K,11-u-nails,1,ok,2,3,2,2,2,1,0.8,0,0,0,0,0,,1,1,65.69,795,3424
M10-GLM-4.7-Flash-Q6_K,11-u-nails,2,ok,3,3,2,3,2,1,0.8,0,0,0,0,0,,1,1,68.3,795,3574
M10-GLM-4.7-Flash-Q6_K,11-u-nails,3,ok,3,3,2,3,2,1,0.8,0,0,0,0,0,,1,1,68.31,795,3574
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,1,ok,3,3,2,3,2,1,0.8,0,0,0,0,0,,1,1,67.65,757,3528
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,2,ok,3,4,3,3,2,1,0.6,0,0,0,0,0,,0,1,51.87,757,2744
M10-GLM-4.7-Flash-Q6_K,12-parikmaherskaya-volokolamskaya,3,ok,3,4,3,3,2,1,0.6,0,0,0,0,0,,0,1,51.83,757,2744
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,1,ok,1,2,1,1,1,1,0.9,0,0,0,0,0,,0,1,57.86,952,3013
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,2,ok,1,2,1,1,1,1,0.95,0,0,0,0,0,,0,1,62.52,952,3272
M10-GLM-4.7-Flash-Q6_K,15-cvety-bez-povoda,3,ok,1,2,1,1,1,1,0.95,0,0,0,0,0,,0,1,62.58,952,3272
M10-GLM-4.7-Flash-Q6_K,16-kameliya,1,ok,3,5,3,5,3,1,0.9,0,0,0,0,0,,0,1,74.98,1119,3850
M10-GLM-4.7-Flash-Q6_K,16-kameliya,2,ok,3,5,4,5,3,1,0.95,0,0,0,0,0,,0,1,45.07,1119,2375
M10-GLM-4.7-Flash-Q6_K,16-kameliya,3,ok,3,5,4,5,3,1,0.95,0,0,0,0,0,,0,1,45.01,1119,2375
M10-GLM-4.7-Flash-Q6_K,17-ip-electronics,1,ok,2,3,2,2,2,1,0.9,0,0,0,0,0,,1,1,53.67,1114,2778
M10-GLM-4.7-Flash-Q6_K,18-service69,2,ok,2,4,2,1,2,1,0.9,0,0,0,0,0,,2,1,47.11,958,2486
M10-GLM-4.7-Flash-Q6_K,18-service69,3,ok,2,4,2,1,2,1,0.9,0,0,0,0,0,,2,1,47.05,958,2486
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,1,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,0,1,62.06,1102,3201
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,2,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,1,1,51.63,1102,2711
M10-GLM-4.7-Flash-Q6_K,19-stecko-service,3,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,1,1,51.6,1102,2711
M10-GLM-4.7-Flash-Q6_K,20-agym,1,ok,3,5,4,4,2,1,0.9,0,0,0,0,0,,0,1,43.88,1039,2287
M10-GLM-4.7-Flash-Q6_K,20-agym,2,ok,4,5,4,5,4,1,0.9,0,0,0,0,0,,0,1,60.32,1039,3178
M10-GLM-4.7-Flash-Q6_K,20-agym,3,ok,3,5,4,5,2,1,0.9,0,0,0,0,0,,1,1,45.57,1039,2458
M10-GLM-4.7-Flash-Q6_K,21-my-fit,2,ok,3,4,4,3,3,1,0.85,0,0,0,0,0,,3,1,46.25,1012,2492
M10-GLM-4.7-Flash-Q6_K,21-my-fit,3,ok,3,4,4,3,3,1,0.85,0,0,0,0,0,,3,1,46.17,1012,2492
M10-GLM-4.7-Flash-Q6_K,22-lizavet,1,ok,2,2,2,2,2,1,0.8,0,0,0,0,0,,0,1,70.38,834,3656
M10-GLM-4.7-Flash-Q6_K,23-tver-kuhni,1,ok,2,2,1,1,1,1,0.8,0,0,0,0,0,,0,1,67.98,769,3544
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,2,ok,2,2,2,1,2,1,0.8,0,0,0,0,0,,0,1,47.36,774,2512
M10-GLM-4.7-Flash-Q6_K,24-rustekstil,3,ok,2,2,2,1,2,1,0.8,0,0,0,0,0,,0,1,47.27,774,2512
M11-gemma-4-31B-it-Q6_K,01-ponchik,1,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,1,1,94.11,975,1480
M12-T-pro-it-2.1-Q6_K,01-ponchik,1,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",6,1,26.11,927,383
M12-T-pro-it-2.1-Q6_K,01-ponchik,2,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",6,1,22.67,927,383
M12-T-pro-it-2.1-Q6_K,01-ponchik,3,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",6,1,22.67,927,383
M12-T-pro-it-2.1-Q6_K,02-tabella-bakery,1,format_fail,2,2,2,2,,1,0.7,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,18.61,751,295
M12-T-pro-it-2.1-Q6_K,02-tabella-bakery,2,format_fail,2,2,2,2,,1,0.7,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,17.09,751,295
M12-T-pro-it-2.1-Q6_K,02-tabella-bakery,3,format_fail,2,2,2,2,,1,0.7,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,17.12,751,295
M12-T-pro-it-2.1-Q6_K,03-surf-coffee,1,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,25.68,897,399
M12-T-pro-it-2.1-Q6_K,03-surf-coffee,2,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",4,1,21.71,897,367
M12-T-pro-it-2.1-Q6_K,03-surf-coffee,3,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",4,1,21.67,897,367
M12-T-pro-it-2.1-Q6_K,04-buna-buna,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,2,1,17.41,777,273
M12-T-pro-it-2.1-Q6_K,04-buna-buna,2,ok,2,2,2,1,1,1,0.85,0,0,0,0,0,,1,1,16.24,777,278
M12-T-pro-it-2.1-Q6_K,04-buna-buna,3,ok,2,2,2,1,1,1,0.85,0,0,0,0,0,,1,1,16.18,777,278
M12-T-pro-it-2.1-Q6_K,05-moncafe,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,19.3,816,303
M12-T-pro-it-2.1-Q6_K,05-moncafe,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,17.74,816,303
M12-T-pro-it-2.1-Q6_K,05-moncafe,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,17.72,816,303
M12-T-pro-it-2.1-Q6_K,06-breakfast-band,1,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,18.83,826,290
M12-T-pro-it-2.1-Q6_K,06-breakfast-band,2,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,16.99,826,290
M12-T-pro-it-2.1-Q6_K,06-breakfast-band,3,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,17.05,826,290
M12-T-pro-it-2.1-Q6_K,07-ru-service,1,ok,1,1,1,1,1,1,0.9,0,0,0,0,0,,0,1,18.54,738,295
M12-T-pro-it-2.1-Q6_K,07-ru-service,2,format_fail,1,1,1,1,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,22.29,738,381
M12-T-pro-it-2.1-Q6_K,07-ru-service,3,format_fail,1,1,1,1,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,22.23,738,381
M12-T-pro-it-2.1-Q6_K,08-avtobat,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,16.66,775,260
M12-T-pro-it-2.1-Q6_K,08-avtobat,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,21.48,775,366
M12-T-pro-it-2.1-Q6_K,08-avtobat,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,21.57,775,366
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,1,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,1,1,24.4,1020,368
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,2,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,1,1,22.12,1020,368
M12-T-pro-it-2.1-Q6_K,09-geometriya-koles,3,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,1,1,22.15,1020,368
M12-T-pro-it-2.1-Q6_K,10-krizanta,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,15.24,760,235
M12-T-pro-it-2.1-Q6_K,10-krizanta,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,13.8,760,235
M12-T-pro-it-2.1-Q6_K,10-krizanta,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,13.86,760,235
M12-T-pro-it-2.1-Q6_K,11-u-nails,1,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,19.2,785,300
M12-T-pro-it-2.1-Q6_K,11-u-nails,2,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,18.09,785,306
M12-T-pro-it-2.1-Q6_K,11-u-nails,3,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,18.19,785,306
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,18.95,742,299
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,14.37,742,244
M12-T-pro-it-2.1-Q6_K,12-parikmaherskaya-volokolamskaya,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,14.41,742,244
M12-T-pro-it-2.1-Q6_K,13-salon-radul,1,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,0,1,26.92,1008,409
M12-T-pro-it-2.1-Q6_K,13-salon-radul,2,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,0,1,24.08,1008,401
M12-T-pro-it-2.1-Q6_K,13-salon-radul,3,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,0,1,24.1,1008,401
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,2,1,20.98,971,313
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,2,1,19.26,971,322
M12-T-pro-it-2.1-Q6_K,14-milliard-roz,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,2,1,19.29,971,322
M12-T-pro-it-2.1-Q6_K,15-cvety-bez-povoda,1,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,22.44,919,342
M12-T-pro-it-2.1-Q6_K,15-cvety-bez-povoda,2,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,20.42,919,342
M12-T-pro-it-2.1-Q6_K,15-cvety-bez-povoda,3,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",3,1,20.39,919,342
M12-T-pro-it-2.1-Q6_K,16-kameliya,1,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",7,1,35.74,1099,546
M12-T-pro-it-2.1-Q6_K,16-kameliya,2,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",5,1,28.07,1099,465
M12-T-pro-it-2.1-Q6_K,16-kameliya,3,format_fail,3,3,3,3,,1,0.9,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",5,1,28.05,1099,465
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,1,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,4,1,28.42,1110,427
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,2,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,4,1,26.28,1110,435
M12-T-pro-it-2.1-Q6_K,17-ip-electronics,3,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,4,1,26.19,1110,435
M12-T-pro-it-2.1-Q6_K,18-service69,1,ok,2,2,2,1,1,1,0.85,0,0,0,0,0,,2,1,20.87,947,313
M12-T-pro-it-2.1-Q6_K,18-service69,2,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,2,1,22.13,947,369
M12-T-pro-it-2.1-Q6_K,18-service69,3,ok,2,2,2,1,1,1,0.9,0,0,0,0,0,,2,1,22.12,947,369
M12-T-pro-it-2.1-Q6_K,19-stecko-service,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,23.32,1101,343
M12-T-pro-it-2.1-Q6_K,19-stecko-service,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,20.66,1101,343
M12-T-pro-it-2.1-Q6_K,19-stecko-service,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,20.67,1101,343
M12-T-pro-it-2.1-Q6_K,20-agym,1,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,4,1,26.28,1023,395
M12-T-pro-it-2.1-Q6_K,20-agym,2,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,4,1,23.81,1023,395
M12-T-pro-it-2.1-Q6_K,20-agym,3,ok,3,3,3,3,2,1,0.9,0,0,0,0,0,,4,1,23.76,1023,395
M12-T-pro-it-2.1-Q6_K,21-my-fit,1,ok,3,3,3,3,1,1,0.9,0,0,0,0,0,,1,1,21.45,964,320
M12-T-pro-it-2.1-Q6_K,21-my-fit,2,ok,3,3,3,3,1,1,0.9,0,0,0,0,0,,2,1,20.19,964,337
M12-T-pro-it-2.1-Q6_K,21-my-fit,3,ok,3,3,3,3,1,1,0.9,0,0,0,0,0,,2,1,20.22,964,337
M12-T-pro-it-2.1-Q6_K,22-lizavet,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,1,1,16.98,804,261
M12-T-pro-it-2.1-Q6_K,22-lizavet,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,2,1,19.25,804,324
M12-T-pro-it-2.1-Q6_K,22-lizavet,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,2,1,19.25,804,324
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,4,1,23.88,757,372
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,2,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,2,1,16.9,757,286
M12-T-pro-it-2.1-Q6_K,23-tver-kuhni,3,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,2,1,16.93,757,286
M12-T-pro-it-2.1-Q6_K,24-rustekstil,1,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,19.55,764,305
M12-T-pro-it-2.1-Q6_K,24-rustekstil,2,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,18.89,764,318
M12-T-pro-it-2.1-Q6_K,24-rustekstil,3,format_fail,2,2,2,1,,1,0.8,0,0,0,0,0,"criteria.data_driven: ожидалось целое 1–5, получено None",2,1,18.89,764,318
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,1,1,47.63,859,3136
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,2,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,1,1,47.52,859,3136
M13-Qwen3.5-35B-A3B-UD-Q6_K,05-moncafe,3,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,1,1,47.26,859,3136
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,1,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,0,1,40.86,861,2685
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,2,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,0,1,40.72,861,2685
M13-Qwen3.5-35B-A3B-UD-Q6_K,06-breakfast-band,3,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,0,1,40.5,861,2685
gemma-7b-it.Q8_0.gguf,01-ponchik,1,format_fail,3,2,2,2,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.58,982,337
gemma-7b-it.Q8_0.gguf,02-tabella-bakery,2,format_fail,2,1,2,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,5.47,793,326
gemma-7b-it.Q8_0.gguf,02-tabella-bakery,3,format_fail,2,1,2,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.4,793,322
gemma-7b-it.Q8_0.gguf,03-surf-coffee,1,format_fail,2,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,7.3,959,391
gemma-7b-it.Q8_0.gguf,03-surf-coffee,2,format_fail,2,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.63,959,334
gemma-7b-it.Q8_0.gguf,03-surf-coffee,3,format_fail,2,2,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,4.86,959,288
gemma-7b-it.Q8_0.gguf,04-buna-buna,1,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.86,820,373
gemma-7b-it.Q8_0.gguf,04-buna-buna,2,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.19,820,308
gemma-7b-it.Q8_0.gguf,04-buna-buna,3,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,4.72,820,280
gemma-7b-it.Q8_0.gguf,05-moncafe,1,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.5,862,347
gemma-7b-it.Q8_0.gguf,05-moncafe,2,format_fail,2,1,2,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.6,862,392
gemma-7b-it.Q8_0.gguf,05-moncafe,3,format_fail,2,2,1,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.06,862,361
gemma-7b-it.Q8_0.gguf,06-breakfast-band,1,format_fail,1,1,0,0,0,0,0,0,0,0,1,1,"criteria.process_automation: ожидалось целое 1–5, получено 0; criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.97,863,312
gemma-7b-it.Q8_0.gguf,06-breakfast-band,2,format_fail,2,2,1,0,0,0,0.6,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,7.69,863,455
gemma-7b-it.Q8_0.gguf,06-breakfast-band,3,format_fail,2,2,1,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.96,863,413
gemma-7b-it.Q8_0.gguf,07-ru-service,1,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.11,783,327
gemma-7b-it.Q8_0.gguf,07-ru-service,2,format_fail,2,1,2,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.49,783,386
gemma-7b-it.Q8_0.gguf,07-ru-service,3,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.55,783,331
gemma-7b-it.Q8_0.gguf,08-avtobat,3,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.49,814,326
gemma-7b-it.Q8_0.gguf,09-geometriya-koles,1,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,4.82,1073,237
gemma-7b-it.Q8_0.gguf,09-geometriya-koles,2,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,5.4,1073,320
gemma-7b-it.Q8_0.gguf,09-geometriya-koles,3,format_fail,2,3,2,2,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.81,1073,344
gemma-7b-it.Q8_0.gguf,10-krizanta,1,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,4.98,802,259
gemma-7b-it.Q8_0.gguf,10-krizanta,2,format_fail,2,1,2,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.41,802,380
gemma-7b-it.Q8_0.gguf,10-krizanta,3,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,9.93,802,590
gemma-7b-it.Q8_0.gguf,11-u-nails,1,format_fail,2,2,1,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,4.73,825,244
gemma-7b-it.Q8_0.gguf,11-u-nails,2,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.37,825,320
gemma-7b-it.Q8_0.gguf,11-u-nails,3,format_fail,2,2,1,0,0,0,0.75,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.03,825,299
gemma-7b-it.Q8_0.gguf,12-parikmaherskaya-volokolamskaya,1,format_fail,2,2,1,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.95,786,377
gemma-7b-it.Q8_0.gguf,12-parikmaherskaya-volokolamskaya,2,format_fail,2,1,2,0,0,0,0,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.03,786,298
gemma-7b-it.Q8_0.gguf,12-parikmaherskaya-volokolamskaya,3,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,9.52,786,566
gemma-7b-it.Q8_0.gguf,13-salon-radul,1,format_fail,3,2,1,1,0,0,0.5,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,8.5,1065,457
gemma-7b-it.Q8_0.gguf,13-salon-radul,2,format_fail,3,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,5.02,1065,297
gemma-7b-it.Q8_0.gguf,13-salon-radul,3,format_fail,2,2,1,0,0,0,0.7,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.53,1065,387
gemma-7b-it.Q8_0.gguf,14-milliard-roz,1,format_fail,2,1,1,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,7.45,1025,395
gemma-7b-it.Q8_0.gguf,14-milliard-roz,2,format_fail,2,2,1,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.38,1025,319
gemma-7b-it.Q8_0.gguf,14-milliard-roz,3,format_fail,2,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.36,1025,316
gemma-7b-it.Q8_0.gguf,15-cvety-bez-povoda,1,format_fail,2,2,1,0,0,0,0.7,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.71,983,297
gemma-7b-it.Q8_0.gguf,15-cvety-bez-povoda,2,format_fail,2,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.35,983,317
gemma-7b-it.Q8_0.gguf,15-cvety-bez-povoda,3,format_fail,2,1,2,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.4,983,379
gemma-7b-it.Q8_0.gguf,16-kameliya,1,format_fail,3,2,1,2,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.97,1147,351
gemma-7b-it.Q8_0.gguf,16-kameliya,2,format_fail,2,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.18,1147,366
gemma-7b-it.Q8_0.gguf,16-kameliya,3,format_fail,3,3,2,2,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,8.41,1147,497
gemma-7b-it.Q8_0.gguf,17-ip-electronics,1,format_fail,3,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.1,1158,305
gemma-7b-it.Q8_0.gguf,17-ip-electronics,3,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,8.15,1158,482
gemma-7b-it.Q8_0.gguf,18-service69,1,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.45,995,338
gemma-7b-it.Q8_0.gguf,18-service69,2,format_fail,2,2,1,0,0,0,0.5,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.56,995,388
gemma-7b-it.Q8_0.gguf,18-service69,3,format_fail,2,3,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,7.82,995,460
gemma-7b-it.Q8_0.gguf,19-stecko-service,1,format_fail,2,2,1,0,0,0,0.7,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.12,1142,312
gemma-7b-it.Q8_0.gguf,19-stecko-service,2,format_fail,2,2,1,0,0,0,0.6,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,5.36,1142,317
gemma-7b-it.Q8_0.gguf,19-stecko-service,3,format_fail,2,3,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,15.13,1142,891
gemma-7b-it.Q8_0.gguf,20-agym,1,format_fail,3,2,2,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,8.96,1075,483
gemma-7b-it.Q8_0.gguf,20-agym,2,format_fail,3,2,1,0,0,0,0.6,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,5.27,1075,312
gemma-7b-it.Q8_0.gguf,20-agym,3,format_fail,2,2,1,1,0,0,0.8,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,7.35,1075,435
gemma-7b-it.Q8_0.gguf,21-my-fit,1,format_fail,2,3,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",1,1,6.65,1034,346
gemma-7b-it.Q8_0.gguf,21-my-fit,2,format_fail,2,3,2,1,1,0,0.8,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,7.43,1034,441
gemma-7b-it.Q8_0.gguf,21-my-fit,3,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,7.48,1034,443
gemma-7b-it.Q8_0.gguf,22-lizavet,1,format_fail,2,1,2,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.94,859,315
gemma-7b-it.Q8_0.gguf,22-lizavet,2,format_fail,2,2,1,0,0,0,0.8,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,6.67,859,397
gemma-7b-it.Q8_0.gguf,22-lizavet,3,format_fail,2,1,2,0,0,0,0.6,0,0,0,1,1,"criteria.ecommerce_online_services: ожидалось целое 1–5, получено 0; criteria.data_driven: ожидалось целое 1–5, получено 0; criteria.ai_usage: ожидалось целое 1–5, получено 0",0,1,5.46,859,326
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,01-ponchik,1,ok,3,3,2,3,2,1,1,0,0,0,0,0,,1,1,4.54,812,290
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,01-ponchik,2,ok,3,3,2,3,2,1,1,0,0,0,0,0,,1,1,3.92,812,293
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,01-ponchik,3,ok,3,3,2,3,2,1,1,0,0,0,0,0,,1,1,3.92,812,293
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,02-tabella-bakery,1,ok,3,3,2,3,1,1,0.8,0,0,0,0,0,,0,1,2.58,656,173
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,02-tabella-bakery,2,ok,3,3,2,3,1,1,0.8,0,0,0,0,0,,0,1,2.28,656,173
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,02-tabella-bakery,3,ok,3,3,2,3,1,1,0.8,0,0,0,0,0,,0,1,2.27,656,173
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,03-surf-coffee,1,ok,3,4,3,3,2,1,1,0,0,0,0,0,,1,1,4.19,800,285
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,03-surf-coffee,2,ok,3,4,2,3,2,1,1,0,0,0,0,0,,1,1,3.82,800,286
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,03-surf-coffee,3,ok,3,4,2,3,2,1,1,0,0,0,0,0,,1,1,3.82,800,286
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,04-buna-buna,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,3.38,678,233
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,04-buna-buna,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,3.08,678,233
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,04-buna-buna,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,3.08,678,233
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,05-moncafe,1,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,3,1,3.45,714,234
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,05-moncafe,2,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,3,1,3.1,714,234
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,05-moncafe,3,ok,2,2,2,2,1,1,0.8,0,0,0,0,0,,3,1,3.1,714,234
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,06-breakfast-band,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,3.7,727,253
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,06-breakfast-band,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,3.36,727,253
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,06-breakfast-band,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,3,1,3.35,727,253
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,07-ru-service,1,ok,1,1,1,1,1,1,1,0,0,0,0,0,,3,1,4.24,651,300
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,07-ru-service,2,ok,1,1,1,1,1,1,1,0,0,0,0,0,,3,1,3.91,651,296
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,07-ru-service,3,ok,1,1,1,1,1,1,1,0,0,0,0,0,,3,1,3.91,651,296
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,08-avtobat,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,0,1,3.12,681,213
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,08-avtobat,2,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,0,1,2.79,681,211
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,08-avtobat,3,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,0,1,2.78,681,211
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,09-geometriya-koles,1,ok,3,3,2,3,2,1,1,0,0,0,0,0,,2,1,4.26,884,286
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,09-geometriya-koles,2,ok,3,3,2,3,2,1,1,0,0,0,0,0,,2,1,3.84,884,286
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,09-geometriya-koles,3,ok,3,3,2,3,2,1,1,0,0,0,0,0,,2,1,3.84,884,286
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,10-krizanta,1,ok,2,2,2,1,1,1,0.7,0,0,0,0,0,,0,1,3.13,664,214
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,10-krizanta,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.38,664,181
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,10-krizanta,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.38,664,181
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,11-u-nails,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.86,683,194
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,11-u-nails,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.71,683,205
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,11-u-nails,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.71,683,205
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,12-parikmaherskaya-volokolamskaya,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.75,653,189
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,12-parikmaherskaya-volokolamskaya,2,ok,2,2,1,1,1,1,0.7,0,0,0,0,0,,0,1,2.43,653,185
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,12-parikmaherskaya-volokolamskaya,3,ok,2,2,1,1,1,1,0.7,0,0,0,0,0,,0,1,2.43,653,185
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,13-salon-radul,1,ok,2,2,2,1,1,1,1,0,0,0,0,0,,1,1,4.36,887,292
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,13-salon-radul,2,ok,2,2,2,1,1,1,1,0,0,0,0,0,,2,1,4.02,887,299
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,13-salon-radul,3,ok,2,2,2,1,1,1,1,0,0,0,0,0,,2,1,4.02,887,299
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,14-milliard-roz,1,ok,2,2,1,1,1,1,1,0,0,0,0,0,,3,1,4.02,848,269
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,14-milliard-roz,2,ok,2,2,1,1,1,1,1,0,0,0,0,0,,3,1,3.42,848,255
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,14-milliard-roz,3,ok,2,2,1,1,1,1,1,0,0,0,0,0,,3,1,3.42,848,255
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,15-cvety-bez-povoda,1,ok,2,2,1,1,1,1,1,0,0,0,0,0,,1,1,2.76,804,177
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,15-cvety-bez-povoda,2,ok,2,2,1,1,1,1,1,0,0,0,0,0,,1,1,2.35,804,177
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,15-cvety-bez-povoda,3,ok,2,2,1,1,1,1,1,0,0,0,0,0,,1,1,2.35,804,177
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,16-kameliya,2,ok,3,4,3,4,2,1,1,0,0,0,0,0,,1,1,5.46,948,403
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,16-kameliya,3,ok,3,4,3,4,2,1,1,0,0,0,0,0,,1,1,5.47,948,403
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,17-ip-electronics,1,ok,2,3,2,2,2,1,1,0,0,0,0,0,,4,1,4.67,950,311
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,17-ip-electronics,2,ok,2,3,2,2,2,1,1,0,0,0,0,0,,4,1,4.0,950,297
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,17-ip-electronics,3,ok,2,3,2,2,2,1,1,0,0,0,0,0,,4,1,4.0,950,297
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,19-stecko-service,1,ok,2,2,2,1,1,1,1,0,0,0,0,0,,4,1,4.83,930,323
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,19-stecko-service,2,ok,2,2,2,1,1,1,1,0,0,0,0,0,,4,1,4.24,930,314
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,19-stecko-service,3,ok,2,2,2,1,1,1,1,0,0,0,0,0,,4,1,4.23,930,314
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,20-agym,1,ok,3,4,3,3,3,1,1,0,0,0,0,0,,3,1,4.15,885,275
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,20-agym,2,ok,3,4,3,3,3,1,1,0,0,0,0,0,,3,1,4.15,885,309
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,20-agym,3,ok,3,4,3,3,3,1,1,0,0,0,0,0,,3,1,4.15,885,309
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,21-my-fit,1,ok,3,3,2,2,2,1,1,0,0,0,0,0,,3,1,4.43,854,299
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,21-my-fit,2,ok,3,3,2,2,2,1,1,0,0,0,0,0,,3,1,4.01,854,299
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,21-my-fit,3,ok,3,3,2,2,2,1,1,0,0,0,0,0,,3,1,4.01,854,299
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,22-lizavet,1,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.77,701,184
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,22-lizavet,2,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.85,701,215
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,22-lizavet,3,ok,2,2,2,1,1,1,0.8,0,0,0,0,0,,0,1,2.84,701,215
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,23-tver-kuhni,1,ok,1,2,1,1,1,1,0.5,0,0,0,0,0,,4,1,3.97,661,266
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,23-tver-kuhni,2,ok,1,1,1,1,1,1,0.5,0,0,0,0,0,,4,1,4.12,661,312
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,23-tver-kuhni,3,ok,1,1,1,1,1,1,0.5,0,0,0,0,0,,4,1,4.12,661,312
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,24-rustekstil,1,ok,1,1,1,1,1,1,1,0,0,0,0,0,,4,1,4.19,671,294
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,24-rustekstil,2,ok,1,1,1,1,1,1,1,0,0,0,0,0,,4,1,3.83,671,290
YandexGPT-5-Lite-8B-instruct-Q4_K_M.gguf,24-rustekstil,3,ok,1,1,1,1,1,1,1,0,0,0,0,0,,4,1,3.84,671,290"""

df_summary_comprehensive = pd.read_csv(io.StringIO(_CSV_DF_SUMMARY_COMPREHENSIVE_BODY))
df_summary_comprehensive.shape
df_summary_comprehensive.head()


,model,dossier,run,status,level,online_presence,process_automation,ecommerce_online_services,data_driven,ai_usage,confidence,n_evidence,has_level_rationale,has_evidence,zero_ai_usage,zero_data_driven,errors,warnings_count,attempts,total_s,prompt_tokens,completion_tokens
0,M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,1,format_fail,3,4,3,4,1.0,0,1.0,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получе...",7,1,6.42,1992,535
1,M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,2,format_fail,3,4,3,4,1.0,0,1.0,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получе...",7,1,4.18,1992,478
2,M6-GigaChat3.1-10B-A1.8B-q8_0,01-ponchik,3,format_fail,3,4,3,4,1.0,0,1.0,0,0,0,1,0,"criteria.ai_usage: ожидалось целое 1–5, получе...",7,1,4.18,1992,478
3,M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,1,format_fail,2,3,2,3,0.0,0,1.0,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, пол...",3,1,3.55,1819,372
4,M6-GigaChat3.1-10B-A1.8B-q8_0,02-tabella-bakery,2,format_fail,2,2,1,2,0.0,0,1.0,0,0,0,1,1,"criteria.data_driven: ожидалось целое 1–5, пол...",2,1,2.54,1819,290


## 3. Быстрый осмотр данных

### 3.1 Статусы прогонов по моделям


In [5]:
df_summary.pivot_table(
    index="model", columns="status", values="run", aggfunc="count", fill_value=0
)

status,format_fail,no_json,ok
model,,,
M10-GLM-4.7-Flash-Q6_K,3,21,48
M11-gemma-4-31B-it-Q6_K,0,0,1
M12-T-pro-it-2.1-Q6_K,26,0,46
M13-Qwen3.5-35B-A3B-UD-Q6_K,3,63,6
M6-GigaChat3.1-10B-A1.8B-q8_0,72,0,0
M7-gpt-oss-20b-Q8_0,2,2,68
M8-saiga_gemma3_12b-Q8_0,31,0,41
M9-gemma-4-12b-it-Q8_0,0,3,13
gemma-7b-it.Q8_0.gguf,65,1,0


### 3.2 Производительность (среднее по прогонам)


In [6]:
(df_summary.groupby("model")[["total_s", "prompt_tokens", "completion_tokens"]]
    .mean().round(2).sort_values("total_s"))

,total_s,prompt_tokens,completion_tokens
model,,,
M6-GigaChat3.1-10B-A1.8B-q8_0,3.48,1956.21,378.88
gemma-7b-it.Q8_0.gguf,6.66,947.50,380.36
M8-saiga_gemma3_12b-Q8_0,17.69,939.53,1354.54
M12-T-pro-it-2.1-Q6_K,20.76,886.71,338.22
M7-gpt-oss-20b-Q8_0,20.82,950.12,1803.14
M13-Qwen3.5-35B-A3B-UD-Q6_K,60.56,929.88,3997.21
M10-GLM-4.7-Flash-Q6_K,63.54,903.75,3326.03
M9-gemma-4-12b-it-Q8_0,66.18,875.44,2481.38
M11-gemma-4-31B-it-Q6_K,94.11,975.00,1480.00


### 3.3 Стабильность уровня (число различных значений `level` за 3 прогона; 1 = полностью стабильно)


In [7]:
ok = df_summary.loc[df_summary["ok"]]
stab = ok.groupby(["model", "dossier"])["level"].nunique()
print(f"Доля полностью стабильных пар (модель, досье): {(stab == 1).mean():.1%}  ({(stab == 1).sum()}/{len(stab)})")
stab.value_counts().rename_axis("число уровней").rename("пар")

Доля полностью стабильных пар (модель, досье): 71.7%  (66/92)


число уровней
1    66
2    26
Name: пар, dtype: int64

---

Источник данных: прогоны LLM-эксперимента (temperature = 0, 3 прогона на досье, JSON-ответ с валидацией схемы). Csv отчёты встроены в ipynb для удобства хранения и представления.